# VFM Choke M1 Physics Tests

Private run of `vfmchoke` (m1_tests.py). The package source is written by the cells below, so this notebook is exactly the code that ran.


In [ ]:
import os
os.makedirs('benchmarks', exist_ok=True)
os.makedirs('src/vfmchoke', exist_ok=True)
os.makedirs('tests', exist_ok=True)


In [ ]:
%%writefile src/vfmchoke/__init__.py
"""vfmchoke: a general few-shot virtual flow meter for wellhead chokes, with choke control and leak attribution."""
__version__ = '0.1.0'


In [ ]:
%%writefile src/vfmchoke/choke.py
"""Wellhead choke models.

Mechanistic (after Sachdeva et al. 1986): homogeneous no-slip two-phase flow through a restriction, liquid
incompressible, free gas expanding isentropically from the upstream pressure P1 to y*P1. The mass flux is

    G(y) = sqrt(2 P1 rho_m2^2 [(1-x)(1-y) v_L + x k/(k-1) (v_G1 - y v_G2)]),   v_G2 = v_G1 y^(-1/k),

with x the free-gas mass fraction at upstream conditions. Flow is critical (choked) below the ratio y_c where G
peaks: there the rate no longer depends on the downstream pressure. Sachdeva's closed-form y_c uses a polytropic
exponent; here y_c is the exact maximiser of G, which keeps the rate continuous at the critical boundary.

Empirical (Gilbert family): P_wh = A q GLR^B / S^C in field units (psig, stb/d, scf/stb, 64ths of an inch),
valid for critical flow only.
"""
import numpy as np

from . import pvt

Y_MIN = 1e-3
CD_NOMINAL = 0.80
GOLDEN_ITERS = 48              # interval shrinks to ~1e-10 of [Y_MIN, 1]

GILBERT_FAMILY = {             # name: (A, B, C) in P_wh[psig] = A * q * GLR**B / S**C
    'gilbert': (10.00, 0.546, 1.89),
    'ros': (17.40, 0.500, 2.00),
    'baxendell': (9.56, 0.546, 1.93),
    'achong': (3.82, 0.650, 1.88),
    'pilehvari': (46.67, 0.313, 2.11),
}


def trim(opening, gamma=1.0):
    """Effective-area fraction of a choke trim at an opening in [0, 1]: opening**gamma.
    gamma = 1 is a linear trim; gamma > 1 approximates equal-percentage trims."""
    return np.clip(np.asarray(opening, dtype=float), 0.0, 1.0) ** gamma


def flux_shape(y, x, v_l, v_g1, k=pvt.K_GAS):
    """G**2 / (2 P1): depends only on the pressure ratio y and the upstream specific volumes and quality."""
    y = np.clip(np.asarray(y, dtype=float), 1e-12, 1.0)
    v_g2 = v_g1 * y ** (-1.0 / k)
    rho_m2 = 1.0 / (x * v_g2 + (1.0 - x) * v_l)
    energy = (1.0 - x) * (1.0 - y) * v_l + x * k / (k - 1.0) * (v_g1 - y * v_g2)
    return rho_m2 ** 2 * np.maximum(energy, 0.0)


def critical_ratio_golden(x, v_l, v_g1, k=pvt.K_GAS, iters=GOLDEN_ITERS):
    """Reference: the flux maximiser by golden-section search on [Y_MIN, 1] (kept for tests)."""
    shape = np.broadcast(np.asarray(x), np.asarray(v_l), np.asarray(v_g1)).shape
    lo = np.full(shape, Y_MIN)
    hi = np.ones(shape)
    g = (np.sqrt(5.0) - 1.0) / 2.0
    for _ in range(iters):
        c = hi - g * (hi - lo)
        d = lo + g * (hi - lo)
        left = flux_shape(c, x, v_l, v_g1, k) >= flux_shape(d, x, v_l, v_g1, k)
        hi = np.where(left, d, hi)
        lo = np.where(left, lo, c)
    return 0.5 * (lo + hi)


def critical_ratio(x, v_l, v_g1, k=pvt.K_GAS, iters=32):          # 2^-32 of [Y_MIN, 1]: ~2e-10
    """Pressure ratio P2/P1 at which the mass flux peaks; below it the choke is critical.

    With D(y) = x v_G1 y^(-1/k) + (1-x) v_L (= 1/rho_m2) and E(y) the energy term of flux_shape, dE/dy = -D, so
    d(E/D^2)/dy = -g(y)/D^3 with g = D^2 - (2/k) E x v_G1 y^(-1/k-1). The flux rises while g < 0 and falls once g > 0;
    the maximiser is the root of g, found by bisection (one evaluation per step). If g >= 0 already at Y_MIN (no
    free gas) the flux only falls, and the ratio is Y_MIN."""
    x, v_l, v_g1 = np.broadcast_arrays(np.asarray(x, float), np.asarray(v_l, float), np.asarray(v_g1, float))

    def g(y):
        a = y ** (-1.0 / k)
        d = x * v_g1 * a + (1.0 - x) * v_l
        e = (1.0 - x) * (1.0 - y) * v_l + x * k / (k - 1.0) * v_g1 * (1.0 - y * a)
        return d * d - (2.0 / k) * e * x * v_g1 * a / y

    lo = np.full(x.shape, Y_MIN)
    hi = np.ones(x.shape)
    none = g(lo) >= 0
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        rising = g(mid) < 0
        lo = np.where(rising, mid, lo)
        hi = np.where(rising, hi, mid)
    return np.where(none, Y_MIN, 0.5 * (lo + hi))


def mass_rate(p1, p2, t1, fluid, opening, d_choke, cd=CD_NOMINAL, trim_gamma=1.0, k=pvt.K_GAS, info=False, st=None):
    """Total mass rate (kg/s) through a choke of bean diameter d_choke (m) at an opening in [0, 1].
    st: the upstream pvt.State when the caller already has it."""
    p1 = np.asarray(p1, dtype=float)
    p2 = np.asarray(p2, dtype=float)
    st = pvt.state(p1, t1, fluid) if st is None else st
    v_l = 1.0 / st.rho_l
    v_g1 = 1.0 / st.rho_g
    yc = critical_ratio(st.x, v_l, v_g1, k)
    y = np.clip(p2 / np.maximum(p1, 1.0), 0.0, 1.0)
    y_eff = np.maximum(y, yc)
    area = np.pi / 4.0 * np.asarray(d_choke, dtype=float) ** 2 * trim(opening, trim_gamma)
    m = cd * area * np.sqrt(2.0 * p1 * flux_shape(y_eff, st.x, v_l, v_g1, k))
    m = np.where(p2 >= p1, 0.0, m)
    if info:
        return m, {'y': y, 'yc': yc, 'critical': y <= yc, 'x': st.x}
    return m


def std_rates(m, fluid):
    """Stock-tank oil, gas and water rates (Sm3/s) carried by a total mass rate m (kg/s) of a stream."""
    q_l = np.asarray(m, dtype=float) / pvt.mass_per_liquid_sc(fluid)
    q_o = (1.0 - fluid.wc) * q_l
    return q_o, fluid.gor * q_o, fluid.wc * q_l


def gilbert_rate(p_wh_psig, s64, glr_scf_stb, name='gilbert'):
    """Liquid rate (stb/d) from a Gilbert-family correlation; field units, critical flow."""
    a, b, c = GILBERT_FAMILY[name]
    return (np.asarray(p_wh_psig, dtype=float) * np.asarray(s64, dtype=float) ** c
            / (a * np.maximum(np.asarray(glr_scf_stb, dtype=float), 1e-9) ** b))


In [ ]:
%%writefile src/vfmchoke/control.py
"""Choke control on the calibrated VFM. Advisory setpoints only: an operator and the control system decide.

The twin of a well is its design deliverability (reservoir pressure, productivity, tubing; corrected by labels
when there are any) with the calibrated VFM as its choke: the physics rate at nominal discharge coefficient and
vendor trim, times exp(f + b0) * opening**b1. Twin and truth share one solver, so a setpoint found on the twin can be
applied to the truth field as is.
"""
from dataclasses import fields, replace

import numpy as np
from scipy.optimize import minimize

from . import network, pvt, well


def replicate(w, n):
    """Wells of shape (W,) as n identical groups, shape (n, W)."""
    km, ke = w._vfm_terms()
    kw = {f.name: np.broadcast_to(np.asarray(getattr(w, f.name)), (n,) + np.shape(w.p_res)).copy()
          for f in fields(w) if f.name not in ('fluid', 'k_mult', 'k_exp')}
    fl = pvt.Fluid(*(np.broadcast_to(getattr(w.fluid, f.name), (n,) + np.shape(w.p_res)).copy()
                     for f in fields(w.fluid)))
    return well.Wells(fluid=fl, k_mult=np.broadcast_to(km, (n,) + np.shape(w.p_res)).copy(),
                      k_exp=np.broadcast_to(ke, (n,) + np.shape(w.p_res)).copy(), **kw)


def replicate_line(line, g, n):
    return network.Line(*(np.full(n, float(np.asarray(getattr(line, f.name))[g])) for f in fields(line)))


def replicate_table(t, n):
    return {k: np.broadcast_to(v, (n,) + v.shape).copy() for k, v in t.items()}


def rates(w, table, openings, line, iters=28):
    """Network solve for stacked candidate opening vectors (n, W). Returns per-candidate totals and well rates."""
    n = openings.shape[0]
    out = network.solve(replicate(w, n), replicate_table(table, n), openings, replicate_line(line, 0, n),
                        iters=iters)
    return {'q_o': out['q_o'].sum(1), 'q_g': out['q_g'].sum(1), 'q_w': out['q_w'].sum(1), 'q_l': out['q_l'],
            'p_man': out['p_man'], 'p_wh': out['p_wh']}


def single_well_setpoint(w1, table1, p2, q_target, u_lo=0.05, u_hi=1.0, iters=24):
    """Opening at which the twin well delivers q_target (Sm3/s liquid) against downstream pressure p2.
    Returns (opening, achievable) where achievable is False when even full opening falls short."""
    q_full = float(np.ravel(well.solve_rate(w1, table1, np.asarray(p2), np.asarray(u_hi))['q_l'])[0])
    if q_full <= q_target:
        return u_hi, False
    lo, hi = u_lo, u_hi
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        q = float(np.ravel(well.solve_rate(w1, table1, np.asarray(p2), np.asarray(mid))['q_l'])[0])
        lo, hi = (mid, hi) if q < q_target else (lo, mid)
    return 0.5 * (lo + hi), True


def scale_heuristic(w_mean, w_hi, table, line, u0, gas_cap, water_cap, iters=18):
    """Operator baseline: scale every open choke by one factor until the cautious twin meets both caps."""
    lo, hi = 0.0, 1.0 / max(float(u0.max()), 1e-6)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        r = rates(w_hi, table, np.clip(u0 * mid, 0, 1)[None, :], line)
        ok = (r['q_g'][0] <= gas_cap) and (r['q_w'][0] <= water_cap)
        lo, hi = (mid, hi) if ok else (lo, mid)
    return np.clip(u0 * lo, 0, 1)


def optimize_group(w_mean, w_hi, table, line, u_start, open_mask, gas_cap, water_cap, h=0.02, maxiter=30):
    """Maximise twin oil over the openings of the open wells, subject to the cautious twin meeting the caps.
    Gradients by batched finite differences: all perturbed candidates go through one network solve."""
    idx = np.nonzero(open_mask)[0]
    cache = {}

    def evaluate(x):
        key = tuple(np.round(x, 6))
        if key in cache:
            return cache[key]
        base = u_start.copy()
        base[idx] = x
        cands = [base]
        for j in idx:
            c = base.copy()
            c[j] = min(c[j] + h, 1.0) if c[j] + h <= 1.0 else c[j] - h
            cands.append(c)
        cands = np.array(cands)
        rm = rates(w_mean, table, cands, line)
        rh = rates(w_hi, table, cands, line)
        steps = (cands[1:, idx] - base[idx]).diagonal()
        f0 = (-rm['q_o'][0], gas_cap - rh['q_g'][0], water_cap - rh['q_w'][0])
        grads = (-(rm['q_o'][1:] - rm['q_o'][0]) / steps, -(rh['q_g'][1:] - rh['q_g'][0]) / steps,
                 -(rh['q_w'][1:] - rh['q_w'][0]) / steps)
        cache[key] = (f0, grads)
        return cache[key]

    scale = max(abs(float(rates(w_mean, table, u_start[None, :], line)['q_o'][0])), 1e-9)
    res = minimize(lambda x: evaluate(x)[0][0] / scale, u_start[idx], jac=lambda x: evaluate(x)[1][0] / scale,
                   method='SLSQP', bounds=[(0.05, 1.0)] * len(idx),
                   constraints=[{'type': 'ineq', 'fun': lambda x: evaluate(x)[0][1] / gas_cap,
                                 'jac': lambda x: evaluate(x)[1][1] / gas_cap},
                                {'type': 'ineq', 'fun': lambda x: evaluate(x)[0][2] / max(water_cap, 1e-9),
                                 'jac': lambda x: evaluate(x)[1][2] / max(water_cap, 1e-9)}],
                   options={'maxiter': maxiter, 'ftol': 1e-6})
    u = u_start.copy()
    u[idx] = np.clip(res.x, 0.05, 1.0)
    return u, {'success': bool(res.success), 'nit': int(res.nit), 'message': str(res.message)}


def cautious(w, z_sd):
    """The same twin with its choke multiplier raised by z standard deviations (upper bound for constraints)."""
    km, ke = w._vfm_terms()
    return replace(w, k_mult=km * np.exp(z_sd), k_exp=ke)


In [ ]:
%%writefile src/vfmchoke/edge.py
"""Edge inference: numpy only, no LightGBM at run time.

- The pooled correction is exported from LightGBM to flat tree arrays and evaluated by array traversal.
- The choke physics is the same numpy code as everywhere else.
- For setpoint optimisation the twin is shipped as lookup tables: each well's liquid rate on a grid of opening and
  downstream pressure, and the flowline's required inlet pressure on a grid of total mass rate. One optimizer step
  is then a short bisection over table lookups.
"""
import io

import numpy as np

from . import choke, hybrid, network, pvt, well


def export_trees(lgbm):
    """Flatten a fitted LGBMRegressor into arrays: per node feature, threshold, children (negative = leaf),
    leaf values, default-left flag. Numerical splits only (all our features are numeric)."""
    dump = lgbm.booster_.dump_model()
    feats, thr, left, right, dleft, miss, leaf_vals, roots = [], [], [], [], [], [], [], []
    miss_code = {'None': 0, 'Zero': 1, 'NaN': 2}

    def walk(node):
        if 'leaf_index' in node or 'split_feature' not in node:
            leaf_vals.append(node.get('leaf_value', 0.0))
            return -len(leaf_vals)                     # leaf k is encoded as -(k+1)
        i = len(feats)
        feats.append(node['split_feature'])
        thr.append(node['threshold'])
        dleft.append(bool(node.get('default_left', True)))
        miss.append(miss_code.get(node.get('missing_type', 'None'), 0))
        left.append(0)
        right.append(0)
        left[i] = walk(node['left_child'])
        right[i] = walk(node['right_child'])
        return i

    for t in dump['tree_info']:
        roots.append(walk(t['tree_structure']))
    return {'feature': np.array(feats, np.int32), 'threshold': np.array(thr, np.float64),
            'left': np.array(left, np.int32), 'right': np.array(right, np.int32),
            'default_left': np.array(dleft, bool), 'missing': np.array(miss, np.int8),
            'leaf': np.array(leaf_vals, np.float64),
            'roots': np.array(roots, np.int32), 'feature_names': np.array(dump['feature_names'])}


def predict_trees(arr, x):
    """Sum of tree outputs for rows x (n, n_features), traversing all trees at once."""
    x = np.atleast_2d(np.asarray(x, dtype=float))
    n, t = x.shape[0], arr['roots'].shape[0]
    node = np.broadcast_to(arr['roots'], (n, t)).copy()
    out = np.zeros(n)
    active = node >= 0
    out += np.where(active, 0.0, arr['leaf'][np.where(active, 0, -node - 1)]).sum(1)
    rows = np.broadcast_to(np.arange(n)[:, None], (n, t))
    while active.any():
        nd = np.where(active, node, 0)
        v = x[rows, arr['feature'][nd]]
        mt = arr['missing'][nd]
        nan = np.isnan(v)
        # LightGBM: missing_type None turns NaN into 0; NaN sends NaN to the default side; Zero also sends ~0 there
        to_default = ((mt == 2) & nan) | ((mt == 1) & (nan | (np.abs(np.nan_to_num(v)) <= 1e-35)))
        v = np.where(nan & (mt == 0), 0.0, v)
        go_left = np.where(to_default, arr['default_left'][nd], v <= arr['threshold'][nd])
        nxt = np.where(go_left, arr['left'][nd], arr['right'][nd])
        node = np.where(active, nxt, node)
        newly_leaf = active & (node < 0)
        out += np.where(newly_leaf, arr['leaf'][np.where(newly_leaf, -node - 1, 0)], 0.0).sum(1)
        active = active & (node >= 0)
    return out


def save(path, arrays, well_params):
    buf = {f'tree_{k}': v for k, v in arrays.items()}
    buf.update({f'well_{k}': np.asarray(v) for k, v in well_params.items()})
    np.savez_compressed(path, **buf)


def load(path):
    z = np.load(path, allow_pickle=False)
    trees = {k[5:]: z[k] for k in z.files if k.startswith('tree_')}
    wp = {k[5:]: z[k] for k in z.files if k.startswith('well_')}
    return trees, wp


def vfm_rate(trees, wp, p1, p2, t1, opening):
    """Calibrated VFM mass rate (kg/s) on the edge, numpy only. wp holds per-well design and calibration values:
    api, gas_sg, gor, wc, sal, d_choke, vendor_gamma, b0, b1."""
    fl = pvt.Fluid(wp['api'], wp['gas_sg'], wp['gor'], wp['wc'], wp['sal'])
    m0, x = hybrid.prior_arrays(p1, p2, t1, opening, wp['d_choke'], wp['vendor_gamma'], fl)
    order = [hybrid.FEATURES.index(str(s)) for s in trees['feature_names']]
    f = predict_trees(trees, x[:, order])
    return m0 * np.exp(f + wp['b0'] + wp['b1'] * np.log(np.maximum(np.atleast_1d(opening), 1e-3)))


def twin_tables(w, table, u_grid, p2_grid):
    """Each twin well's liquid rate (Sm3/s) on an (opening, downstream pressure) grid: shape (W, n_u, n_p)."""
    W = np.shape(w.p_res)[0]
    out = np.zeros((W, len(u_grid), len(p2_grid)))
    for a, uu in enumerate(u_grid):
        for b, pp in enumerate(p2_grid):
            out[:, a, b] = well.solve_rate(w, table, np.full(W, pp), np.full(W, uu))['q_l']
    return out


def line_table(line, fluid_mix, t_mix, m_grid):
    """Required manifold pressure for a total mass rate of the commingled stream (no leak)."""
    p_rb = network.section_up(np.full(len(m_grid), float(line.p_sep[0])), m_grid, fluid_mix, t_mix,
                              float(line.d[0]), float(line.depth[0]), 1.0)
    return network.section_up(p_rb, m_grid, fluid_mix, t_mix, float(line.d[0]), float(line.length[0]), 0.0)


def edge_step(rate_tab, u_grid, p2_grid, mls, m_grid, p_req, u_vec, iters=30):
    """One optimizer step on the edge: manifold pressure and well rates for an opening vector, by bisection on
    the manifold pressure with bilinear table lookups."""
    W = rate_tab.shape[0]
    ui = np.interp(u_vec, u_grid, np.arange(len(u_grid)))
    u0 = np.clip(np.floor(ui).astype(int), 0, len(u_grid) - 2)
    wu = ui - u0
    lo, hi = p2_grid[0], p2_grid[-1]
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        pi_ = np.interp(mid, p2_grid, np.arange(len(p2_grid)))
        p0 = min(int(pi_), len(p2_grid) - 2)
        wp_ = pi_ - p0
        r = rate_tab[np.arange(W), u0]
        r1 = rate_tab[np.arange(W), u0 + 1]
        q = ((1 - wu) * ((1 - wp_) * r[:, p0] + wp_ * r[:, p0 + 1])
             + wu * ((1 - wp_) * r1[:, p0] + wp_ * r1[:, p0 + 1]))
        need = np.interp((q * mls).sum(), m_grid, p_req)
        lo, hi = (mid, hi) if mid < need else (lo, mid)
    return 0.5 * (lo + hi), q


def artifact_bytes(arrays, well_params):
    bio = io.BytesIO()
    np.savez_compressed(bio, **{f't_{k}': v for k, v in arrays.items()},
                        **{f'w_{k}': np.asarray(v) for k, v in well_params.items()})
    return bio.getbuffer().nbytes


In [ ]:
%%writefile src/vfmchoke/fewshot.py
"""Per-well calibration in closed form.

Rate model for one well: log m = log m_prior + f(features) + b0 + b1 * log(opening) + noise.
(b0, b1) is a latent pair per well: b0 absorbs a wrong discharge coefficient or bean size, b1 a wrong trim curve.
Its prior N(0, diag(tau0^2, tau1^2)) is learned from training wells (empirical Bayes); b0 may drift slowly
(erosion) as a random walk with variance q0 per day. Labels carry their own noise: MPFM readings wider than
test-separator tests. With no labels the prediction is the general model itself; each label tightens it.

Fluid ratios (water cut in logit space, GOR in log space) are tracked the same way: a scalar random walk that
starts at the design value.
"""
from dataclasses import dataclass, asdict

import numpy as np


@dataclass
class LatentPrior:
    tau0: float = 0.10       # sd of b0 across wells
    tau1: float = 0.10       # sd of b1 across wells
    q0: float = 1e-5         # b0 random-walk variance per day
    sig_mpfm: float = 0.04   # sd of an MPFM label (log scale), model error included
    sig_test: float = 0.02   # sd of a separator-test label (log scale), model error included
    sig_model: float = 0.02  # sd of day-to-day model error at prediction time (log scale)

    def to_dict(self):
        return asdict(self)


def latent_posterior(t_obs, phi_obs, r_obs, sd_obs, prior):
    """Kalman filter over the observations (time order). Returns the state mean (2,), covariance (2, 2) and the
    time of the last observation. phi rows are (1, log opening)."""
    x = np.zeros(2)
    p = np.diag([prior.tau0 ** 2, prior.tau1 ** 2])
    t_last = None
    for t, phi, r, sd in sorted(zip(t_obs, phi_obs, r_obs, sd_obs), key=lambda z: z[0]):
        if t_last is not None:
            p = p + np.diag([prior.q0 * (t - t_last), 0.0])
        h = np.asarray(phi, dtype=float)
        s = h @ p @ h + sd ** 2
        k = p @ h / s
        x = x + k * (r - h @ x)
        p = p - np.outer(k, h) @ p
        t_last = t
    return x, p, t_last


def latent_predict(x, p, t_last, t_pred, phi_pred, prior):
    """Mean and variance of the log-rate correction at prediction times (after the last observation)."""
    phi_pred = np.asarray(phi_pred, dtype=float)
    dt = np.maximum(np.asarray(t_pred, dtype=float) - (t_last if t_last is not None else 0.0), 0.0)
    drift = prior.q0 * dt if t_last is not None else 0.0
    mean = phi_pred @ x
    var = np.einsum('ni,ij,nj->n', phi_pred, p, phi_pred) + drift + prior.sig_model ** 2
    return mean, var


def track(t_obs, y_obs, sd_obs, m0, sd0, q, t_pred):
    """Scalar random-walk Kalman filter: posterior mean at each prediction time (after the observations)."""
    m, v, t_last = m0, sd0 ** 2, 0.0
    for t, y, sd in sorted(zip(t_obs, y_obs, sd_obs), key=lambda z: z[0]):
        v = v + q * max(t - t_last, 0.0)
        k = v / (v + sd ** 2)
        m = m + k * (y - m)
        v = (1 - k) * v
        t_last = t
    return np.full(np.shape(t_pred), m), v


def logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def expit(z):
    return 1.0 / (1.0 + np.exp(-z))


def fit_prior(per_well, sig_floor=0.01):
    """Empirical-Bayes prior from training wells.
    per_well: list of dicts with 'b' (2,), 'cov' (2, 2) from each well's least-squares fit, 'resid_sd' (MPFM),
    'test_resid_sd' (may be nan) and 'b0_windows' (array of b0 fitted on consecutive 30-day windows)."""
    b = np.array([w['b'] for w in per_well])
    cov = np.array([w['cov'] for w in per_well])
    tau2 = np.maximum(b.var(axis=0) - cov[:, [0, 1], [0, 1]].mean(axis=0), 1e-6)
    diffs = np.concatenate([np.diff(w['b0_windows']) for w in per_well if len(w['b0_windows']) > 1] or [np.zeros(1)])
    q0 = max(float(np.var(diffs)) / 30.0 - 2 * float(np.median(cov[:, 0, 0])) / 30.0, 1e-8)
    sig_mpfm = max(float(np.median([w['resid_sd'] for w in per_well])), sig_floor)
    tests = [w['test_resid_sd'] for w in per_well if np.isfinite(w.get('test_resid_sd', np.nan))]
    sig_test = max(float(np.median(tests)) if tests else sig_mpfm / 2, sig_floor)
    return LatentPrior(tau0=float(np.sqrt(tau2[0])), tau1=float(np.sqrt(tau2[1])), q0=q0, sig_mpfm=sig_mpfm,
                       sig_test=sig_test, sig_model=max(sig_test, sig_floor))


def ols_well(phi, r):
    """Least-squares (b0, b1) of one well with its covariance and residual sd."""
    phi = np.asarray(phi, dtype=float)
    r = np.asarray(r, dtype=float)
    if len(r) < 3 or np.ptp(phi[:, 1]) < 1e-6:
        b0 = float(np.mean(r)) if len(r) else 0.0
        sd = float(np.std(r)) if len(r) > 1 else 0.05
        return np.array([b0, 0.0]), np.diag([sd ** 2 / max(len(r), 1), 1.0]), sd
    b, *_ = np.linalg.lstsq(phi, r, rcond=None)
    res = r - phi @ b
    sd = float(np.sqrt(res @ res / max(len(r) - 2, 1)))
    cov = sd ** 2 * np.linalg.inv(phi.T @ phi)
    return b, cov, sd


In [ ]:
%%writefile src/vfmchoke/fleet.py
"""Fleet-level machinery shared by the evaluation, control and leak milestones: label preparation, training the
general model and its empirical-Bayes prior, calibrating one well from k labels, and predicting with intervals.
"""
from dataclasses import dataclass

import numpy as np
import pandas as pd

from . import fewshot, hybrid, network, pvt, units as u

MIN_Q = u.stb_d_to_si(10.0)
LABEL_SD = {'mpfm': {'wc': 0.3, 'gor': 0.08}, 'test': {'wc': 0.1, 'gor': 0.03}}


def prepare(daily, tests, wells):
    """Add design data, mass labels, observed ratios and trailing ratio estimates (what a live system holds)."""
    wells = wells.set_index('well_id') if 'well_id' in wells.columns else wells
    daily = daily.copy()
    for c in ('api', 'gas_sg', 'gor', 'wc', 'sal'):
        daily['design_' + c] = wells.loc[daily.well_id, 'design_' + c].to_numpy()
    des = pvt.Fluid(api=daily.design_api, gas_sg=daily.design_gas_sg, gor=daily.design_gor, wc=daily.design_wc,
                    salinity=daily.design_sal)
    daily['label_m'] = network.std_mass(daily.mpfm_q_o, daily.mpfm_q_g, daily.mpfm_q_w, des)
    daily['true_q_l'] = daily.true_q_o + daily.true_q_w
    daily['flowing'] = (daily.opening > 0) & (daily.true_q_l > MIN_Q) & (daily.label_m > 0)
    daily['obs_wc'] = daily.mpfm_q_w / np.maximum(daily.mpfm_q_o + daily.mpfm_q_w, 1e-12)
    daily['obs_gor'] = daily.mpfm_q_g / np.maximum(daily.mpfm_q_o, 1e-12)
    daily = daily.sort_values(['well_id', 'day']).reset_index(drop=True)
    g = daily.where(daily.flowing).groupby(daily.well_id)
    daily['trail_wc'] = g.obs_wc.transform(lambda s: s.shift(1).rolling(7, min_periods=1).median())
    daily['trail_gor'] = g.obs_gor.transform(lambda s: s.shift(1).rolling(7, min_periods=1).median())
    daily['trail_wc'] = daily.trail_wc.fillna(daily.design_wc).clip(0, 0.98)
    daily['trail_gor'] = daily.trail_gor.fillna(daily.design_gor).clip(1, 3000)

    tests = tests.merge(daily[['well_id', 'day', 'opening', 'p_wh', 't_wh', 'p_man', 'd_choke', 'vendor_gamma']],
                        on=['well_id', 'day'], how='left')
    for c in ('api', 'gas_sg', 'gor', 'wc', 'sal'):
        tests['design_' + c] = wells.loc[tests.well_id, 'design_' + c].to_numpy()
    tdes = pvt.Fluid(api=tests.design_api, gas_sg=tests.design_gas_sg, gor=tests.design_gor, wc=tests.design_wc,
                     salinity=tests.design_sal)
    tests['label_m'] = network.std_mass(tests.test_q_o, tests.test_q_g, tests.test_q_w, tdes)
    tests['obs_wc'] = tests.test_q_w / np.maximum(tests.test_q_o + tests.test_q_w, 1e-12)
    tests['obs_gor'] = tests.test_q_g / np.maximum(tests.test_q_o, 1e-12)
    return daily, tests, wells


def with_est(df, wc, gor):
    out = df.copy()
    out['est_api'], out['est_gas_sg'], out['est_sal'] = df.design_api, df.design_gas_sg, df.design_sal
    out['est_wc'], out['est_gor'] = wc, gor
    return out


SCALE_KS = {'mpfm': (0, 1, 2, 3, 5, 10, 30), 'test': (0, 1, 2, 3)}


@dataclass
class Trained:
    model: hybrid.Hybrid
    prior: fewshot.LatentPrior
    scales: dict = None      # interval scale by regime and number of labels (conformal, from training wells)

    def scale(self, regime, k):
        table = (self.scales or {}).get(regime, {})
        keys = [kk for kk in table if kk <= k]
        return table[max(keys)] if keys else 1.0


def train(tr, tests, n_inner=3, seed=0, interval_scales=False, cal_end=90):
    """Fit the pooled correction on training rows (flowing, MPFM labels) and the empirical-Bayes latent prior from
    out-of-fold residuals. tr: prepared daily rows of training wells; tests: prepared tests (any wells).
    interval_scales: also calibrate the interval width by replaying the label-then-forecast protocol on training
    wells with the inner models that never saw them."""
    trf = with_est(tr, tr.trail_wc, tr.trail_gor)
    m_pr, feats = hybrid.prior(trf, hybrid.fluid_from(trf))
    ok = (m_pr > 0) & np.isfinite(m_pr)
    trf, feats, m_pr = trf[ok], feats[ok], m_pr[ok]
    resid = np.log(trf.label_m.to_numpy() / m_pr)
    inner = (trf.group.to_numpy().astype(int) * 2654435761 % 2 ** 32) % n_inner
    f_oof = np.zeros(len(trf))
    inner_models = []
    for j in range(n_inner):
        mdl = hybrid.Hybrid(n_estimators=300, seed=seed).fit(feats[inner != j], resid[inner != j])
        f_oof[inner == j] = mdl.correction(feats[inner == j])
        inner_models.append(mdl)
    model = hybrid.Hybrid(seed=seed).fit(feats, resid)
    trf = trf.assign(r=resid - f_oof, phi1=np.log(np.maximum(trf.opening.to_numpy(), 1e-3)))
    per_well = []
    tr_tests = tests[tests.well_id.isin(trf.well_id.unique())]
    for wid, gw in trf.groupby('well_id'):
        phi = np.column_stack([np.ones(len(gw)), gw.phi1.to_numpy()])
        b, cov, sd = fewshot.ols_well(phi, gw.r.to_numpy())
        win = [float(np.mean(gw.r.to_numpy()[m] - b[1] * gw.phi1.to_numpy()[m]))
               for m in [(gw.day.to_numpy() // 30) == k for k in range(12)] if m.sum() >= 10]
        tw = tr_tests[tr_tests.well_id == wid]
        t_sd = np.nan
        if len(tw) >= 2:
            twf = with_est(tw, tw.obs_wc.clip(0, 0.98), tw.obs_gor.clip(1, 3000))
            tm, tfe = hybrid.prior(twf, hybrid.fluid_from(twf))
            rr = np.log(tw.label_m.to_numpy() / tm) - model.correction(tfe) - (b[0] + b[1] * np.log(tw.opening.to_numpy()))
            t_sd = float(np.std(rr))
        per_well.append({'b': b, 'cov': cov, 'resid_sd': sd, 'test_resid_sd': t_sd, 'b0_windows': np.array(win)})
    trained = Trained(model=model, prior=fewshot.fit_prior(per_well))
    if interval_scales:
        inner_of_group = {int(gid): int(j) for gid, j in zip(trf.group.to_numpy(), inner)}
        trained.scales = _interval_scales(trf, tests, inner_models, inner_of_group, trained, cal_end, seed)
    return trained


def _interval_scales(trf, tests, inner_models, inner_of_group, trained, cal_end, seed, n_wells=300):
    """Split-conformal width per regime and label count: the 90th percentile of |standardised error| when training
    wells go through the deployment protocol (first k labels before cal_end, stale ratios, forecast after), with
    the inner model that did not see the well. Errors are against MPFM labels, so their noise is in the
    denominator."""
    rng = np.random.default_rng(seed)
    wids = trf.well_id.unique()
    wids = rng.choice(wids, min(n_wells, len(wids)), replace=False)
    lp = trained.prior
    zs = {(reg, k): [] for reg, ks in SCALE_KS.items() for k in ks}
    for wid in wids:
        gw = trf[trf.well_id == wid].sort_values('day')
        ev = gw[gw.day >= cal_end]
        if len(ev) < 5:
            continue
        sub = Trained(model=inner_models[inner_of_group[int(gw.group.iloc[0])]], prior=lp)
        design = gw.iloc[0]
        for reg, ks in SCALE_KS.items():
            src = gw[gw.day < cal_end] if reg == 'mpfm' else \
                tests[(tests.well_id == wid) & (tests.day < cal_end)].sort_values('day')
            for k in ks:
                c = calibrate(src.head(k), design, sub, reg)
                m_hat, lo, hi, _ = predict(ev, c, sub)
                sd = (np.log(hi) - np.log(m_hat)) / 1.645
                z = np.log(ev.label_m.to_numpy() / m_hat) / np.sqrt(sd ** 2 + lp.sig_mpfm ** 2)
                zs[(reg, k)].append(z)
    out = {reg: {} for reg in SCALE_KS}
    for (reg, k), parts in zs.items():
        if parts:
            out[reg][k] = float(max(np.quantile(np.abs(np.concatenate(parts)), 0.90) / 1.645, 1.0))
    return out


@dataclass
class Calibration:
    wc: float
    gor: float
    x: np.ndarray
    p: np.ndarray
    t_last: object
    regime: str = 'mpfm'
    k: int = 0


def calibrate(cal, design_row, trained, regime):
    """Fluid ratios and latent posterior of one well from its calibration labels (k rows, may be empty)."""
    sd = LABEL_SD[regime]
    mw, _ = fewshot.track(cal.day, fewshot.logit(cal.obs_wc.clip(0, 0.98)), [sd['wc']] * len(cal),
                          fewshot.logit(max(float(design_row.design_wc), 1e-3)), 1.5, 1e-4, [0])
    mg, _ = fewshot.track(cal.day, np.log(cal.obs_gor.clip(1, 3000)), [sd['gor']] * len(cal),
                          np.log(float(design_row.design_gor)), 0.15, 1e-5, [0])
    wc, gor = float(fewshot.expit(mw[0])), float(np.exp(mg[0]))
    lp = trained.prior
    if len(cal):
        c = with_est(cal, wc, gor)
        m_c, fe_c = hybrid.prior(c, hybrid.fluid_from(c))
        r = np.log(c.label_m.to_numpy() / m_c) - trained.model.correction(fe_c)
        phi = np.column_stack([np.ones(len(c)), np.log(np.maximum(c.opening.to_numpy(), 1e-3))])
        sd_lab = lp.sig_mpfm if regime == 'mpfm' else lp.sig_test
        x, p, t_last = fewshot.latent_posterior(c.day.to_numpy(), phi, r, np.full(len(c), sd_lab), lp)
    else:
        x, p, t_last = np.zeros(2), np.diag([lp.tau0 ** 2, lp.tau1 ** 2]), None
    return Calibration(wc=wc, gor=gor, x=x, p=p, t_last=t_last, regime=regime, k=int(len(cal)))


def predict(rows, cal, trained, z=1.645):
    """Mass rate with a two-sided interval and the estimated fluid, for rows of one well. The interval uses the
    conformal scale for this regime and label count when the model was trained with interval_scales."""
    e = with_est(rows, cal.wc, cal.gor)
    fl = hybrid.fluid_from(e)
    m0, fe = hybrid.prior(e, fl)
    base = m0 * np.exp(trained.model.correction(fe))
    phi = np.column_stack([np.ones(len(e)), np.log(np.maximum(e.opening.to_numpy(), 1e-3))])
    mean, var = fewshot.latent_predict(cal.x, cal.p, cal.t_last, e.day.to_numpy(), phi, trained.prior)
    sd = np.sqrt(var) * trained.scale(cal.regime, cal.k)
    return base * np.exp(mean), base * np.exp(mean - z * sd), base * np.exp(mean + z * sd), fl


In [ ]:
%%writefile src/vfmchoke/hybrid.py
"""The general choke VFM: a physics prior from design data, corrected by a pooled monotone LightGBM model.

The prior is the mechanistic choke rate at the measured wellhead and manifold pressures, the wellhead temperature,
the opening, the bean size and the vendor trim curve, with the well's estimated fluid ratios and a nominal discharge
coefficient. The correction is a multiplicative factor exp(f(features)) learned across many wells from inputs that
mean the same thing on every well: pressure ratios, gas quality and volume fraction, water cut, GOR, opening.
No well identity enters, so the model applies unchanged to a well it has never seen.

f is constrained to rise with opening and pressure drop and to fall with the pressure ratio, so the corrected rate
keeps the physics' monotonicity and the choke inverse stays a bisection with one answer.
"""
import numpy as np
import pandas as pd

from . import choke, pvt

MONOTONE = {'opening': 1, 'log_opening': 1, 'dp_bar': 1, 'y': -1, 'y_over_yc': -1}


def fluid_from(df, prefix='est_'):
    return pvt.Fluid(api=df[prefix + 'api'].to_numpy(float), gas_sg=df[prefix + 'gas_sg'].to_numpy(float),
                     gor=df[prefix + 'gor'].to_numpy(float), wc=df[prefix + 'wc'].to_numpy(float),
                     salinity=df[prefix + 'sal'].to_numpy(float))


FEATURES = ['y', 'y_over_yc', 'quality', 'gvf', 'opening', 'log_opening', 'd_choke_mm', 'vendor_gamma', 'wc',
            'gor', 'api', 'rho_l', 'p1_bar', 'dp_bar']


def prior_arrays(p1, p2, t1, opening, d_choke, vendor_gamma, fluid, cd=choke.CD_NOMINAL):
    """Numpy-only core of prior(): mass rate (kg/s) and the feature matrix (n, len(FEATURES))."""
    p1, p2, t1, op = (np.atleast_1d(np.asarray(a, dtype=float)) for a in (p1, p2, t1, opening))
    d_choke = np.broadcast_to(np.asarray(d_choke, dtype=float), p1.shape)
    vg = np.broadcast_to(np.asarray(vendor_gamma, dtype=float), p1.shape)
    st = pvt.state(p1, t1, fluid)
    m, info = choke.mass_rate(p1, p2, t1, fluid, op, d_choke, cd, vg, info=True, st=st)
    cols = [info['y'], info['y'] / info['yc'], st.x, st.v_g / (st.v_g + st.v_l), op, np.log(np.maximum(op, 1e-3)),
            d_choke * 1e3, vg, fluid.wc, fluid.gor, fluid.api, st.rho_l, p1 / 1e5, (p1 - p2) / 1e5]
    return m, np.column_stack([np.broadcast_to(c, p1.shape) for c in cols])


def prior(df, fluid, cd=choke.CD_NOMINAL):
    """Physics-prior total mass rate (kg/s) and the transferable feature table.
    df needs p_wh, p_man, t_wh (SI), opening, d_choke (m), vendor_gamma."""
    m, x = prior_arrays(df['p_wh'].to_numpy(float), df['p_man'].to_numpy(float), df['t_wh'].to_numpy(float),
                        df['opening'].to_numpy(float), df['d_choke'].to_numpy(float),
                        df['vendor_gamma'].to_numpy(float), fluid, cd)
    return m, pd.DataFrame(x, columns=FEATURES, index=df.index)


class Hybrid:
    """Pooled correction model. fit() on (features, log(m_label / m_prior)); correction() returns f."""

    def __init__(self, n_estimators=500, learning_rate=0.05, num_leaves=31, min_child_samples=200, seed=0):
        self.params = dict(n_estimators=n_estimators, learning_rate=learning_rate, num_leaves=num_leaves,
                           min_child_samples=min_child_samples, subsample=0.8, subsample_freq=1,
                           colsample_bytree=0.9, random_state=seed, verbose=-1)
        self.model = None
        self.cols = None

    def fit(self, feats, residual, weight=None):
        import lightgbm as lgb
        self.cols = list(feats.columns)
        mono = [MONOTONE.get(c, 0) for c in self.cols]
        self.model = lgb.LGBMRegressor(monotone_constraints=mono, monotone_constraints_method='advanced',
                                       **self.params)
        self.model.fit(feats[self.cols], residual, sample_weight=weight)
        return self

    def correction(self, feats):
        return self.model.predict(feats[self.cols])


In [ ]:
%%writefile src/vfmchoke/leak.py
"""Which source fed one oil slick? A ranked posterior over leak hypotheses for one manifold group.

Hypotheses: no leak; the jumper of well k (each well); the manifold; flowline segments; riser positions.
Evidence, each a log-likelihood:
  1. Phase signature of the mass-balance deficit. The per-phase residual r = sum of well-level rates (MPFM or VFM)
     minus the separator, averaged after the detected onset and with its pre-onset bias removed, should point along
     the leaked fluid's composition: well k's own (oil, gas, water) for its jumper, the commingled mix downstream.
     Profile likelihood over the leak size lambda >= 0.
  2. Pressure signature: manifold and riser-base pressure shifts after onset against each hypothesis' predicted
     sensitivity (from the network model) times lambda.
  3. Slick location: the up-current end of the slick, back-tracked for drift, against each hypothesis' seabed
     position.
  4. Volume consistency: oil released between onset and detection (after weathering) against the slick volume range.
Mass balance localises to the line; separating wells needs their ratios to differ. The posterior says how well the
evidence separates them, rather than forcing a single answer.
"""
from dataclasses import dataclass

import numpy as np


@dataclass
class Hypothesis:
    kind: str          # 'none', 'jumper', 'manifold', 'flowline', 'riser'
    well: int          # slot for jumper hypotheses, else -1
    pos: float         # fraction along flowline / riser
    xy: tuple          # seabed position, m (manifold at the origin, flowline along +x)


def layout(n_slots, line_length, rng):
    """Well positions on a drill-centre ring around the manifold, flowline along +x to the riser base."""
    ang = rng.uniform(0, 2 * np.pi, n_slots)
    rad = rng.uniform(40, 180, n_slots)
    return np.column_stack([rad * np.cos(ang), rad * np.sin(ang)]), np.array([line_length, 0.0])


def hypotheses(open_slots, well_xy, riser_xy, fl_positions=(0.1, 0.3, 0.5, 0.7, 0.9), riser_positions=(0.2, 0.6)):
    hs = [Hypothesis('none', -1, 0.0, (np.nan, np.nan))]
    hs += [Hypothesis('jumper', int(s), 0.0, tuple(well_xy[s] * 0.5)) for s in open_slots]
    hs.append(Hypothesis('manifold', -1, 0.0, (0.0, 0.0)))
    hs += [Hypothesis('flowline', -1, p, (p * riser_xy[0], 0.0)) for p in fl_positions]
    hs += [Hypothesis('riser', -1, p, tuple(riser_xy)) for p in riser_positions]
    return hs


def cusum_onset(x, ref_end, k_sd=0.5, h_sd=6.0):
    """One-sided CUSUM on a deficit series. Baseline mean and sd from x[:ref_end].
    Returns (alarm index or None, onset estimate or None)."""
    mu, sd = np.mean(x[:ref_end]), max(np.std(x[:ref_end]), 1e-12)
    s, start = 0.0, None
    for i in range(ref_end, len(x)):
        s_new = max(0.0, s + (x[i] - mu) / sd - k_sd)
        if s == 0.0 and s_new > 0:
            start = i
        s = s_new
        if s > h_sd:
            return i, start
    return None, None


def phase_loglik(r_mean, r_cov, direction):
    """Max over lambda >= 0 of the Gaussian log-likelihood of r_mean = lambda * direction."""
    inv = np.linalg.inv(r_cov)
    c = np.asarray(direction, dtype=float)
    denom = float(c @ inv @ c)
    if not np.isfinite(denom) or denom <= 0.0:          # no usable direction (e.g. a well shut in all along)
        return -0.5 * float(r_mean @ inv @ r_mean), 0.0
    lam = max(float(c @ inv @ r_mean) / denom, 0.0)
    e = r_mean - lam * c
    return -0.5 * float(e @ inv @ e), lam


def posterior(hs, r_mean, r_cov, compositions, dp_obs, dp_cov, dp_sens, slick_xy, loc_sd, drift_xy,
              released_vol, slick_vol, vol_sd_log=0.7, prior_none=0.2):
    """Combine the evidence into a posterior over hypotheses.
    compositions[i]: phase direction (oil, gas, water) per hypothesis (None for 'none').
    dp_sens[i]: predicted (d p_man, d p_riser_base) per unit leak oil rate for hypothesis i.
    released_vol(i, lam): oil volume released by detection for hypothesis i at leak oil rate lam."""
    out = []
    inv_dp = np.linalg.inv(dp_cov)
    for i, h in enumerate(hs):
        if h.kind == 'none':
            ll_phase = -0.5 * float(r_mean @ np.linalg.inv(r_cov) @ r_mean)
            ll_dp = -0.5 * float(dp_obs @ inv_dp @ dp_obs)
            ll_loc = np.log(1e-3)          # a slick with no subsea source: passing vessel, seep; flat, small
            ll_vol, lam = 0.0, 0.0
        else:
            ll_phase, lam = phase_loglik(r_mean, r_cov, compositions[i])
            e = dp_obs - lam * np.asarray(dp_sens[i])
            ll_dp = -0.5 * float(e @ inv_dp @ e)
            d = np.asarray(slick_xy) - (np.asarray(h.xy) + np.asarray(drift_xy))
            ll_loc = -0.5 * float(d @ d) / loc_sd ** 2
            v = max(released_vol(i, lam), 1e-9)
            ll_vol = -0.5 * ((np.log(v) - np.log(slick_vol)) / vol_sd_log) ** 2
        out.append({'hypothesis': h, 'lambda': lam, 'll_phase': ll_phase, 'll_dp': ll_dp, 'll_loc': ll_loc,
                    'll_vol': ll_vol})
    n_src = len(hs) - 1
    logp = np.array([o['ll_phase'] + o['ll_dp'] + o['ll_loc'] + o['ll_vol']
                     + np.log(prior_none if o['hypothesis'].kind == 'none' else (1 - prior_none) / n_src)
                     for o in out])
    p = np.exp(logp - logp.max())
    p /= p.sum()
    for o, pi in zip(out, p):
        o['posterior'] = float(pi)
    return sorted(out, key=lambda o: -o['posterior'])


In [ ]:
%%writefile src/vfmchoke/mpfm.py
"""Gamma-ray multiphase flow meter (MPFM) emulator: dual-energy densitometer plus venturi.

Densitometer (Beer-Lambert): ln(I0/I)_E = d * sum_i alpha_i * mu_i(E) * rho_i over oil, water and gas, at a low
energy (32 keV, photoelectric-sensitive) and a high energy (356 keV, Compton-dominated). Two attenuations plus
the closure sum(alpha) = 1 give the three phase holdups by a 3x3 linear solve.

Venturi: total mass rate from the differential pressure and the gamma mixture density, so a density error moves
the mass rate by sqrt(rho_meter / rho_true).

Holdups become flow fractions through the meter's drift-flux slip model. Errors emulated: Poisson counting noise,
a stale water salinity in the meter configuration (shifts the low-energy water attenuation and the water cut),
a slip-model mismatch, a venturi discharge-coefficient bias, and multiplicative flow-regime noise.

Mass attenuation coefficients are approximate values (cm2/g) interpolated from NIST XCOM tables; the emulator needs
the contrast between phases and energies, not metrology-grade constants.
"""
from dataclasses import dataclass

import numpy as np

from . import pvt

MU_MASS = {            # cm2/g at (32 keV, 356 keV)
    'oil': (0.245, 0.114),
    'water': (0.348, 0.111),
    'gas': (0.262, 0.125),
    'nacl': (2.00, 0.100),
}


def mu_linear(rho_o, rho_w, rho_g, salinity):
    """Linear attenuation coefficients (1/m), shape (..., 2 energies, 3 phases: oil, water, gas)."""
    sal = np.asarray(salinity, dtype=float)
    out = []
    for e in (0, 1):
        mw = (1.0 - sal) * MU_MASS['water'][e] + sal * MU_MASS['nacl'][e]
        out.append(np.stack(np.broadcast_arrays(MU_MASS['oil'][e] * 0.1 * rho_o, mw * 0.1 * rho_w,
                                                MU_MASS['gas'][e] * 0.1 * rho_g), axis=-1))
    return np.stack(out, axis=-2)


def attenuation(alpha, mu, d):
    """ln(I0/I) at both energies for holdups alpha (..., 3) through a path length d (m)."""
    return np.asarray(d, dtype=float)[..., None] * np.einsum('...ep,...p->...e', mu, alpha)


def invert(a, mu, d):
    """Holdups (..., 3) from attenuations (..., 2): two energies plus closure."""
    m = np.concatenate([mu, np.ones(mu.shape[:-2] + (1, 3))], axis=-2)
    rhs = np.concatenate([a / np.asarray(d, dtype=float)[..., None], np.ones(a.shape[:-1] + (1,))], axis=-1)
    return np.linalg.solve(m, rhs[..., None])[..., 0]


@dataclass
class Config:
    """What the meter believes, and its counting setup. Arrays broadcast against the wells."""
    salinity: np.ndarray            # configured water salinity (NaCl mass fraction)
    c0: float = 1.2                 # slip model distribution parameter
    vd_scale: float = 1.0           # slip model drift-velocity multiplier
    cd_ratio: float = 1.0           # venturi discharge coefficient, meter / true (scalar or array)
    counts: tuple = (2e5, 1e5)      # empty-pipe count rates (1/s) at (32 keV, 356 keV)
    integration_s: float = 86400.0  # averaging time of one reading
    noise_l: float = 0.0            # flow-regime noise on liquid (relative std)
    noise_g: float = 0.0            # flow-regime noise on gas (relative std)


def _flow_from_holdup(alpha_g, m, area, rho_l, rho_g, d, c0, vd_scale):
    """Superficial velocities from gas holdup and total mass flux, inverting the drift-flux slip model."""
    v_d = vd_scale * 0.35 * np.sqrt(9.80665 * d * np.maximum(rho_l - rho_g, 0.0) / rho_l)
    v_m = (m / area - alpha_g * v_d * (rho_g - rho_l)) / (rho_g * alpha_g * c0 + rho_l * (1.0 - alpha_g * c0))
    v_sg = alpha_g * (c0 * v_m + v_d)
    return v_m - v_sg, v_sg


def read(p, t, fluid, q_l, d, cfg, rng=None, true_c0=1.2, true_vd_scale=1.0):
    """One MPFM reading of a well producing q_l (Sm3/s stock-tank liquid) at line conditions (p, t).
    Returns reported standard oil, gas and water rates (Sm3/s) and the meter's holdups."""
    st = pvt.state(p, t, fluid)
    area = np.pi * np.asarray(d, dtype=float) ** 2 / 4.0
    v_sl = q_l * st.v_l / area
    v_sg = q_l * st.v_g / area
    v_d = true_vd_scale * 0.35 * np.sqrt(9.80665 * d * np.maximum(st.rho_l - st.rho_g, 0.0) / st.rho_l)
    alpha_g = np.where(v_sg > 0, v_sg / np.maximum(true_c0 * (v_sl + v_sg) + v_d, 1e-12), 0.0)
    wl = st.v_w / st.v_l
    alpha = np.stack([(1 - alpha_g) * (1 - wl), (1 - alpha_g) * wl, alpha_g], axis=-1)
    mu_true = mu_linear(st.rho_o, st.rho_w, st.rho_g, fluid.salinity)
    a = attenuation(alpha, mu_true, d)
    if rng is not None:
        n0 = np.asarray(cfg.counts) * cfg.integration_s
        n = rng.poisson(n0 * np.exp(-a))
        a = np.log(n0 / np.maximum(n, 1))
    rho_w_meter = pvt.rho_water_sc(cfg.salinity) * np.ones_like(st.rho_w)
    mu_meter = mu_linear(st.rho_o, rho_w_meter, st.rho_g, cfg.salinity)
    al = invert(a, mu_meter, d)
    rho_true = (alpha * np.stack([st.rho_o, st.rho_w, st.rho_g], -1)).sum(-1)
    rho_meter = (al * np.stack([st.rho_o, rho_w_meter, st.rho_g], -1)).sum(-1)
    m_true = q_l * pvt.mass_per_liquid_sc(fluid)
    m_meter = m_true * cfg.cd_ratio * np.sqrt(np.maximum(rho_meter, 1e-6) / rho_true)
    liq_al = np.maximum(al[..., 0] + al[..., 1], 1e-12)
    wl_m = np.clip(al[..., 1] / liq_al, 0.0, 1.0)
    rho_l_m = (1 - wl_m) * st.rho_o + wl_m * rho_w_meter
    v_sl_m, v_sg_m = _flow_from_holdup(np.clip(al[..., 2], 0.0, 0.999), m_meter, area, rho_l_m, st.rho_g, d,
                                       cfg.c0, cfg.vd_scale)
    q_o_line = v_sl_m * area * (1 - wl_m)
    q_w_line = v_sl_m * area * wl_m
    q_g_line = np.maximum(v_sg_m, 0.0) * area
    q_o = q_o_line / st.bo
    q_w = q_w_line
    q_g = q_g_line / st.bg + st.rs * q_o
    if rng is not None and (cfg.noise_l > 0 or cfg.noise_g > 0):
        el = np.exp(rng.normal(0.0, cfg.noise_l, np.shape(q_o)))
        q_o, q_w = q_o * el, q_w * el
        q_g = q_g * np.exp(rng.normal(0.0, cfg.noise_g, np.shape(q_g)))
    return {'q_o': q_o, 'q_g': q_g, 'q_w': q_w, 'alpha': al, 'alpha_true': alpha}


In [ ]:
%%writefile src/vfmchoke/network.py
"""Subsea production network: wells -> chokes -> manifold -> flowline -> riser -> separator, with optional leaks.

Arrays are shaped (G, W): G manifold groups, each with up to W wells (pad unused slots with pi = 0, they never
flow). The manifold pressure of every group is found by bisection: a higher manifold pressure lets less flow in,
while the flowline needs a higher inlet pressure for more flow, so the balance is unique.

Leak kinds: 0 none, 1 jumper of one well (that well's fluid, at manifold pressure), 2 manifold (commingled fluid),
3 flowline at fraction `pos` of its length from the manifold, 4 riser at fraction `pos` of its height from the
base. A leak is a sharp-edged hole (two-phase orifice, choke model at full opening) discharging to seawater
hydrostatic pressure at its depth; nothing leaks where the line pressure is below ambient.
"""
from dataclasses import dataclass, fields

import numpy as np

from . import choke, pvt, units as u, well

NONE, JUMPER, MANIFOLD, FLOWLINE, RISER = 0, 1, 2, 3, 4
CD_HOLE = 0.62


@dataclass
class Line:
    """Flowline and riser of each group, arrays of shape (G,)."""
    length: np.ndarray      # horizontal flowline length, m
    d: np.ndarray           # inner diameter (flowline and riser), m
    depth: np.ndarray       # water depth = riser height, m
    p_sep: np.ndarray       # separator pressure, Pa

    def __post_init__(self):
        for f in fields(self):
            setattr(self, f.name, np.asarray(getattr(self, f.name), dtype=float))


@dataclass
class Leak:
    """One optional leak per group, arrays of shape (G,)."""
    kind: np.ndarray        # NONE, JUMPER, MANIFOLD, FLOWLINE or RISER
    well: np.ndarray        # well slot for JUMPER leaks
    pos: np.ndarray         # fraction along the flowline (FLOWLINE) or up the riser (RISER)
    d_hole: np.ndarray      # equivalent hole diameter, m

    @staticmethod
    def none(g):
        return Leak(kind=np.zeros(g, int), well=np.zeros(g, int), pos=np.zeros(g), d_hole=np.zeros(g))


def section_up(p_down, m, fluid, t, d, length, vertical, n_seg=8, rough=4.5e-5):
    """Upstream pressure of a pipe section from its downstream pressure, marching against the flow."""
    q_l = np.maximum(m, 0.0) / pvt.mass_per_liquid_sc(fluid)
    p = np.array(p_down, dtype=float)
    dz = length / n_seg
    for _ in range(n_seg):
        p = p + well.gradient(p, t, q_l, fluid, d, rough, vertical) * dz
    return p


def ambient(depth):
    return u.P_SC + u.RHO_SEAWATER * u.G * np.asarray(depth, dtype=float)


def hole_rate(p_node, depth_node, t, fluid, d_hole):
    """Two-phase mass rate out of a hole to seawater at its depth."""
    return choke.mass_rate(p_node, ambient(depth_node), t, fluid, 1.0, d_hole, CD_HOLE)


def std_mass(q_o, q_g, q_w, fluid):
    return q_o * pvt.rho_oil_sc(fluid.api) + q_g * pvt.rho_gas_sc(fluid.gas_sg) + q_w * pvt.rho_water_sc(fluid.salinity)


def _evaluate(p_man, wells, table, opening, line, leak, inner_iters=40):
    g_idx = np.arange(p_man.shape[0])
    op = well.solve_rate(wells, table, p_man[:, None] * np.ones_like(opening), opening)
    q_o, q_g, q_w = choke.std_rates(op['m'], wells.fluid)

    # jumper or manifold leak: removed before the flowline, at manifold pressure
    k = np.clip(leak.well, 0, opening.shape[1] - 1)
    jf = wells.fluid.take((g_idx, k))
    t_mix = (op['m'] * op['t_wh']).sum(1) / np.maximum(op['m'].sum(1), 1e-12)
    t_mix = np.where(op['m'].sum(1) > 0, t_mix, wells.t_sea.max(1))
    mf = pvt.mix(q_o, q_g, q_w, wells.fluid, axis=1)
    m_in_all = op['m'].sum(1)
    m_jumper = np.minimum(hole_rate(p_man, line.depth, op['t_wh'][g_idx, k], jf, leak.d_hole), op['m'][g_idx, k])
    m_manifold = np.minimum(hole_rate(p_man, line.depth, t_mix, mf, leak.d_hole), m_in_all)
    m_up_leak = np.where(leak.kind == JUMPER, m_jumper, np.where(leak.kind == MANIFOLD, m_manifold, 0.0))
    leak_fluid_up = pvt.Fluid(*(np.where(leak.kind == JUMPER, getattr(jf, n), getattr(mf, n))
                                for n in ('api', 'gas_sg', 'gor', 'wc', 'salinity')))
    lq_o, lq_g, lq_w = choke.std_rates(m_up_leak, leak_fluid_up)

    # stream entering the flowline
    fq_o = q_o.sum(1) - lq_o
    fq_g = q_g.sum(1) - lq_g
    fq_w = q_w.sum(1) - lq_w
    ff = pvt.mix(np.stack([q_o.sum(1), -lq_o], 1), np.stack([q_g.sum(1), -lq_g], 1),
                 np.stack([q_w.sum(1), -lq_w], 1),
                 pvt.Fluid(*(np.stack([getattr(mf, n), getattr(leak_fluid_up, n)], 1)
                             for n in ('api', 'gas_sg', 'gor', 'wc', 'salinity'))), axis=1)
    m_in = std_mass(fq_o, fq_g, fq_w, ff)

    # flowline and riser carry the inflow to the separator
    p_rb = section_up(line.p_sep, m_in, ff, t_mix, line.d, line.depth, 1.0)
    p_man_implied = section_up(p_rb, m_in, ff, t_mix, line.d, line.length, 0.0)
    p_node = p_rb.copy()
    m_down_leak = np.zeros_like(m_in)
    is_fl, is_rs = leak.kind == FLOWLINE, leak.kind == RISER
    if np.any(is_fl | is_rs):
        # leak node pressure: the hydraulics downstream of the node must carry what the hole does not take
        node_depth = np.where(is_rs, line.depth * (1.0 - leak.pos), line.depth)
        riser_above = line.depth * (1.0 - np.where(is_rs, leak.pos, 0.0))
        fl_after = line.length * (1.0 - np.where(is_fl, leak.pos, 0.0))

        def node_needed(m_down):
            p_top_part = section_up(line.p_sep, m_down, ff, t_mix, line.d, riser_above, 1.0)
            return np.where(is_fl, section_up(p_top_part, m_down, ff, t_mix, line.d, fl_after, 0.0), p_top_part)

        lo, hi = line.p_sep.copy(), line.p_sep + 400e5
        for _ in range(inner_iters):
            mid = 0.5 * (lo + hi)
            m_l = np.minimum(hole_rate(mid, node_depth, t_mix, ff, leak.d_hole), m_in)
            f = mid - node_needed(m_in - m_l)
            lo = np.where(f < 0, mid, lo)
            hi = np.where(f < 0, hi, mid)
        node = 0.5 * (lo + hi)
        leaking = is_fl | is_rs
        m_down_leak = np.where(leaking, np.minimum(hole_rate(node, node_depth, t_mix, ff, leak.d_hole), m_in), 0.0)
        p_node = np.where(leaking, node, p_node)
        # upstream of the node the full inflow flows
        rb_fl = section_up(line.p_sep, m_in - m_down_leak, ff, t_mix, line.d, line.depth, 1.0)
        rb_rs = section_up(node, m_in, ff, t_mix, line.d, line.depth * leak.pos, 1.0)
        man_fl = section_up(node, m_in, ff, t_mix, line.d, line.length * leak.pos, 0.0)
        man_rs = section_up(rb_rs, m_in, ff, t_mix, line.d, line.length, 0.0)
        p_rb = np.where(is_fl, rb_fl, np.where(is_rs, rb_rs, p_rb))
        p_man_implied = np.where(is_fl, man_fl, np.where(is_rs, man_rs, p_man_implied))

    dq_o, dq_g, dq_w = choke.std_rates(m_down_leak, ff)
    leak_m = m_up_leak + m_down_leak
    out = {
        'p_man': p_man, 'p_man_implied': p_man_implied, 'p_riser_base': p_rb, 'p_node': p_node, 't_mix': t_mix,
        'q_l': op['q_l'], 'p_wh': op['p_wh'], 't_wh': op['t_wh'], 'm': op['m'],
        'q_o': q_o, 'q_g': q_g, 'q_w': q_w,
        'leak_m': leak_m,
        'leak_q_o': lq_o + dq_o, 'leak_q_g': lq_g + dq_g, 'leak_q_w': lq_w + dq_w,
        'sep_q_o': fq_o - dq_o, 'sep_q_g': fq_g - dq_g, 'sep_q_w': fq_w - dq_w,
        'sep_fluid': ff, 'm_in': m_in,
    }
    out['sep_m'] = std_mass(out['sep_q_o'], out['sep_q_g'], out['sep_q_w'], ff)
    return out


def solve(wells, table, opening, line, leak=None, iters=40):
    """Operating point of every group. wells/opening shaped (G, W); table from well.vlp_table(wells)."""
    opening = np.asarray(opening, dtype=float)
    leak = leak if leak is not None else Leak.none(opening.shape[0])
    lo = line.p_sep.copy()
    hi = np.maximum(table['p_wh'][..., 0].max(axis=1), line.p_sep + 1e5)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        res = _evaluate(mid, wells, table, opening, line, leak)
        f = mid - res['p_man_implied']
        lo = np.where(f < 0, mid, lo)
        hi = np.where(f < 0, hi, mid)
    return _evaluate(0.5 * (lo + hi), wells, table, opening, line, leak)


In [ ]:
%%writefile src/vfmchoke/pvt.py
"""Black-oil PVT from standard correlations. SI in and out; the correlations run in field units inside.

Standing (1947) solution gas, bubble point and oil formation volume factor; Beggs & Robinson (1975) oil viscosity;
Papay (1968) gas Z factor with Sutton (1985) pseudo-criticals; Lee, Gonzalez & Eakin (1966) gas viscosity;
Brill & Beggs water viscosity. Water formation volume factor is taken as 1 and brine density follows salinity.

A stream is described per standard cubic metre of stock-tank liquid (oil + water), so wells with any water cut in
[0, 1) work, including water-dominated ones.
"""
from dataclasses import dataclass, fields

import numpy as np

from . import units as u

RHO_W_SC_FRESH = 999.012          # kg/m3, fresh water at 60 F
K_GAS = 1.28                      # heat capacity ratio of natural gas


@dataclass
class Fluid:
    """Ratios and phase properties of a produced stream (scalars or broadcastable arrays)."""
    api: np.ndarray        # stock-tank oil gravity, degrees API
    gas_sg: np.ndarray     # gas specific gravity (air = 1)
    gor: np.ndarray        # producing gas-oil ratio, Sm3 gas per Sm3 stock-tank oil (free + dissolved)
    wc: np.ndarray         # water cut: water volume fraction of stock-tank liquid, in [0, 1)
    salinity: np.ndarray   # NaCl mass fraction of the produced water

    def __post_init__(self):
        for f in fields(self):
            setattr(self, f.name, np.asarray(getattr(self, f.name), dtype=float))

    def expand(self, axis=-1):
        """The same fluid with a new axis, so it broadcasts against per-well grids."""
        return Fluid(*(np.expand_dims(getattr(self, f.name), axis) for f in fields(self)))

    def take(self, idx):
        return Fluid(*(np.asarray(getattr(self, f.name))[idx] for f in fields(self)))

    def broadcast_to(self, shape):
        return Fluid(*(np.broadcast_to(getattr(self, f.name), shape).copy() for f in fields(self)))


def rho_oil_sc(api):
    return RHO_W_SC_FRESH * u.api_to_sg(api)


def rho_gas_sc(gas_sg):
    return np.asarray(gas_sg, dtype=float) * u.RHO_AIR_SC


def rho_water_sc(salinity):
    """Brine density at standard conditions from the NaCl mass fraction (linear fit, within ~1% up to 20 wt%)."""
    return RHO_W_SC_FRESH * (1.0 + 0.695 * np.asarray(salinity, dtype=float))


def mass_per_liquid_sc(f):
    """Total mass (kg) carried with one Sm3 of stock-tank liquid: oil, all its gas, and water."""
    return (1.0 - f.wc) * (rho_oil_sc(f.api) + f.gor * rho_gas_sc(f.gas_sg)) + f.wc * rho_water_sc(f.salinity)


def bubble_point(t, f):
    """Standing (1947) bubble-point pressure (Pa) for a solution GOR equal to the producing GOR."""
    rsb = np.maximum(u.si_to_scf_stb(f.gor), 1e-6)
    t_f = u.k_to_f(t)
    pb = 18.2 * ((rsb / f.gas_sg) ** 0.83 * 10.0 ** (0.00091 * t_f - 0.0125 * f.api) - 1.4)
    return u.psia_to_pa(np.maximum(pb, u.P_SC / u.PSI))


def solution_gor(p, t, f):
    """Standing (1947) solution GOR (Sm3/Sm3), capped at the producing GOR."""
    p_psia = np.maximum(u.pa_to_psia(p), u.P_SC / u.PSI)
    t_f = u.k_to_f(t)
    rs = f.gas_sg * ((p_psia / 18.2 + 1.4) * 10.0 ** (0.0125 * f.api - 0.00091 * t_f)) ** (1.0 / 0.83)
    return np.minimum(u.scf_stb_to_si(rs), f.gor)


def oil_fvf(rs, t, f):
    """Standing (1947) oil formation volume factor (rm3/Sm3)."""
    rs_scf = u.si_to_scf_stb(rs)
    t_f = u.k_to_f(t)
    return 0.9759 + 0.00012 * (rs_scf * np.sqrt(f.gas_sg / u.api_to_sg(f.api)) + 1.25 * t_f) ** 1.2


def z_factor(p, t, gas_sg):
    """Papay (1968) Z factor with Sutton (1985) pseudo-critical properties."""
    gas_sg = np.asarray(gas_sg, dtype=float)
    tpc = 169.2 + 349.5 * gas_sg - 74.0 * gas_sg ** 2          # R
    ppc = 756.8 - 131.0 * gas_sg - 3.6 * gas_sg ** 2           # psia
    tpr = u.k_to_r(t) / tpc
    ppr = u.pa_to_psia(p) / ppc
    z = 1.0 - 3.52 * ppr / 10.0 ** (0.9813 * tpr) + 0.274 * ppr ** 2 / 10.0 ** (0.8157 * tpr)
    return np.clip(z, 0.3, 1.2)


def gas_fvf(p, t, gas_sg):
    """Gas formation volume factor (rm3/Sm3)."""
    return u.P_SC / np.asarray(p, dtype=float) * np.asarray(t, dtype=float) / u.T_SC * z_factor(p, t, gas_sg)


def gas_density(p, t, gas_sg):
    return rho_gas_sc(gas_sg) / gas_fvf(p, t, gas_sg)


def dead_oil_viscosity(t, api):
    """Beggs & Robinson (1975) dead-oil viscosity, Pa s."""
    t_f = np.maximum(u.k_to_f(t), 60.0)
    x = t_f ** -1.163 * np.exp(6.9824 - 0.04658 * np.asarray(api, dtype=float))
    return (10.0 ** x - 1.0) * 1e-3


def live_oil_viscosity(rs, t, api):
    """Beggs & Robinson (1975) live-oil viscosity, Pa s."""
    rs_scf = u.si_to_scf_stb(rs)
    a = 10.715 * (rs_scf + 100.0) ** -0.515
    b = 5.44 * (rs_scf + 150.0) ** -0.338
    return a * (dead_oil_viscosity(t, api) * 1e3) ** b * 1e-3


def gas_viscosity(p, t, gas_sg):
    """Lee, Gonzalez & Eakin (1966) gas viscosity, Pa s."""
    m = np.asarray(gas_sg, dtype=float) * u.M_AIR
    t_r = u.k_to_r(t)
    rho = gas_density(p, t, gas_sg) / 1000.0                  # g/cm3
    k = (9.4 + 0.02 * m) * t_r ** 1.5 / (209.0 + 19.0 * m + t_r)
    x = 3.5 + 986.0 / t_r + 0.01 * m
    y = 2.4 - 0.2 * x
    return 1e-4 * k * np.exp(x * rho ** y) * 1e-3


def water_viscosity(t):
    """Brill & Beggs water viscosity, Pa s."""
    t_f = u.k_to_f(t)
    return np.exp(1.003 - 1.479e-2 * t_f + 1.982e-5 * t_f ** 2) * 1e-3


@dataclass
class State:
    """In-situ properties at (p, t), per Sm3 of stock-tank liquid where noted."""
    rs: np.ndarray       # solution GOR, Sm3/Sm3
    bo: np.ndarray       # oil FVF, rm3/Sm3
    bg: np.ndarray       # gas FVF, rm3/Sm3
    rho_o: np.ndarray    # live-oil density, kg/m3
    rho_g: np.ndarray    # gas density, kg/m3
    rho_w: np.ndarray    # brine density, kg/m3
    rho_l: np.ndarray    # oil + water liquid density, kg/m3
    mu_l: np.ndarray     # liquid viscosity (volume-weighted), Pa s
    mu_g: np.ndarray     # gas viscosity, Pa s
    v_o: np.ndarray      # in-situ oil volume per Sm3 liquid, m3
    v_w: np.ndarray      # in-situ water volume per Sm3 liquid, m3
    v_g: np.ndarray      # in-situ free-gas volume per Sm3 liquid, m3
    m_l: np.ndarray      # liquid mass per Sm3 liquid, kg
    m_g: np.ndarray      # free-gas mass per Sm3 liquid, kg
    x: np.ndarray        # free-gas mass fraction (quality)

    @property
    def v_l(self):
        return self.v_o + self.v_w


def state(p, t, f):
    """Phase split and properties of the stream at pressure p (Pa) and temperature t (K)."""
    p = np.maximum(np.asarray(p, dtype=float), u.P_SC)
    t = np.asarray(t, dtype=float)
    rs = solution_gor(p, t, f)
    bo = oil_fvf(rs, t, f)
    bg = gas_fvf(p, t, f.gas_sg)
    rho_osc, rho_gsc, rho_wsc = rho_oil_sc(f.api), rho_gas_sc(f.gas_sg), rho_water_sc(f.salinity)
    oil = 1.0 - f.wc
    free = np.maximum(f.gor - rs, 0.0)
    m_o = oil * (rho_osc + rs * rho_gsc)
    m_w = f.wc * rho_wsc
    m_g = oil * free * rho_gsc
    v_o = oil * bo
    v_w = f.wc * np.ones_like(bo)
    v_g = oil * free * bg
    rho_o = (rho_osc + rs * rho_gsc) / bo
    rho_g = rho_gsc / bg
    m_l = m_o + m_w
    v_l = v_o + v_w
    rho_l = m_l / v_l
    mu_o = live_oil_viscosity(rs, t, f.api)
    mu_w = water_viscosity(t)
    mu_l = (v_o * mu_o + v_w * mu_w) / v_l
    return State(rs=rs, bo=bo, bg=bg, rho_o=rho_o, rho_g=rho_g, rho_w=rho_wsc * np.ones_like(bo), rho_l=rho_l,
                 mu_l=mu_l, mu_g=gas_viscosity(p, t, f.gas_sg), v_o=v_o, v_w=v_w, v_g=v_g, m_l=m_l, m_g=m_g,
                 x=m_g / (m_g + m_l))


def mix(q_o, q_g, q_w, fluids, axis=-1):
    """Commingled stream of several wells: rate-weighted ratios and phase properties (mass-exact)."""
    so, sg_, sw = q_o.sum(axis), q_g.sum(axis), q_w.sum(axis)
    oil_sg = (q_o * u.api_to_sg(fluids.api)).sum(axis) / np.maximum(so, 1e-30)
    gas_sg = (q_g * fluids.gas_sg).sum(axis) / np.maximum(sg_, 1e-30)
    sal = (q_w * fluids.salinity).sum(axis) / np.maximum(sw, 1e-30)
    return Fluid(api=u.sg_to_api(np.where(so > 0, oil_sg, 0.85)),
                 gas_sg=np.where(sg_ > 0, gas_sg, 0.7),
                 gor=np.where(so > 0, sg_ / np.maximum(so, 1e-30), 0.0),
                 wc=np.clip(sw / np.maximum(so + sw, 1e-30), 0.0, 0.999),
                 salinity=np.where(sw > 0, sal, 0.0))


In [ ]:
%%writefile src/vfmchoke/simfield.py
"""SimField: a synthetic population of subsea wells on manifolds, simulated day by day.

Each group has one manifold and one flowline + riser to a separator. Each well has a reservoir, a tubing string,
and a choke whose discharge coefficient and trim exponent are hidden (the vendor trim type is known, the as-built
deviation is not). Over time reservoir pressure declines, water cut rises, GOR drifts, the choke erodes, operators
move chokes and shut wells in, and produced-water salinity drifts while the gamma meter keeps its configured value.
Every well has a gamma MPFM (daily readings with their own errors) and a test-separator well test about monthly.
Leaks can be scheduled per group to build slick scenarios.

Truth columns (prefixed true_ or hidden) are for evaluation only; models see design data, sensors and labels.
"""
from dataclasses import dataclass

import numpy as np
import pandas as pd

from . import choke, mpfm, network, pvt, units as u, well


@dataclass
class Population:
    wells: well.Wells          # truth at day 0, arrays (G, W)
    mask: np.ndarray           # real well slots
    line: network.Line
    vendor_gamma: np.ndarray   # trim exponent on the vendor datasheet
    decline: np.ndarray        # reservoir pressure decline, Pa/day
    wc_rate: np.ndarray        # water-cut rise per day
    gor_drift: np.ndarray      # GOR log-drift per day
    erosion: np.ndarray        # relative Cd growth per day
    sal_target: np.ndarray     # salinity the produced water drifts toward (seawater breakthrough)
    sal_rate: np.ndarray       # fraction of the gap closed per day
    meter_sal: np.ndarray      # salinity configured in the MPFM (stale)
    meter_cd: np.ndarray       # MPFM venturi Cd ratio
    meter_d: np.ndarray        # MPFM throat/pipe diameter, m
    test_phase: np.ndarray     # day offset of the monthly test cycle
    well_id: np.ndarray        # global id per slot (-1 for padding)


def _choke_size(w, line_p_sep, frac=0.6):
    """Bean diameter that passes about `frac` of open flow at full opening against a typical manifold pressure."""
    q_des = frac * well.aof(w.p_res, w.p_b, w.pi)
    m_des = q_des * pvt.mass_per_liquid_sc(w.fluid)
    p1 = np.maximum(line_p_sep + 25e5, 40e5)
    m_ref = choke.mass_rate(p1, 0.5 * p1, u.c_to_k(60.0), w.fluid, 1.0, 0.0254, choke.CD_NOMINAL)
    d = 0.0254 * np.sqrt(m_des / np.maximum(m_ref, 1e-9))
    return np.round(np.clip(d, 0.008, 0.064) / (u.INCH / 64.0)) * (u.INCH / 64.0)


def sample_population(n_groups=200, max_wells=10, seed=0, choke_frac=0.6,
                      line_d=(0.1524, 0.2032, 0.254, 0.3048), depth=(30, 400), p_sep=(8e5, 25e5),
                      length=(1000, 15000), flow_margin=30e5):
    """Defaults reproduce the M2 v1 population; the keyword ranges let a variant change the network."""
    rng = np.random.default_rng(seed)
    g, w = n_groups, max_wells
    shape = (g, w)
    n_wells = rng.integers(4, w + 1, g)
    mask = np.arange(w)[None, :] < n_wells[:, None]
    fl = pvt.Fluid(api=rng.uniform(25, 45, shape), gas_sg=rng.uniform(0.62, 0.9, shape),
                   gor=np.exp(rng.uniform(np.log(50), np.log(350), shape)),
                   wc=np.where(rng.random(shape) < 0.3, 0.0, rng.uniform(0.0, 0.5, shape)),
                   salinity=rng.uniform(0.02, 0.15, shape))
    vendor_gamma = rng.choice([1.0, 1.6], shape)
    line = network.Line(length=rng.uniform(*length, g), d=rng.choice(list(line_d), g),
                        depth=rng.uniform(*depth, g), p_sep=rng.uniform(*p_sep, g))
    wells = well.Wells(p_res=rng.uniform(200e5, 360e5, shape), t_res=u.c_to_k(rng.uniform(70, 130, shape)),
                       pi=np.exp(rng.uniform(np.log(2), np.log(30), shape)) * well.PI_FIELD,
                       depth=rng.uniform(1500, 3200, shape), d_tub=rng.choice([0.0762, 0.1005, 0.1213], shape),
                       d_choke=np.full(shape, 0.0254),
                       cd=np.clip(rng.normal(0.82, 0.04, shape), 0.68, 0.95),
                       trim_gamma=vendor_gamma * np.exp(rng.normal(0.0, 0.08, shape)),
                       t_sea=np.repeat(u.c_to_k(rng.uniform(4, 25, g))[:, None], w, 1),
                       q_t=np.full(shape, 0.002), fluid=fl)
    # every real well must be able to flow: shut-in wellhead pressure at least 30 bar above the separator
    for _ in range(4):
        shut_in = well.vlp_table(wells, n_q=8)['p_wh'][..., 0]
        deficit = (line.p_sep[:, None] + flow_margin) - shut_in
        if not np.any(mask & (deficit > 0)):
            break
        wells.p_res = np.where(mask & (deficit > 0), wells.p_res + deficit + 5e5, wells.p_res)
    wells.d_choke = _choke_size(wells, line.p_sep[:, None], choke_frac)
    # padding slots: no inflow at all
    wells.p_res = np.where(mask, wells.p_res, u.P_SC)
    wells.pi = np.where(mask, wells.pi, 1e-30)
    ids = np.cumsum(mask.ravel()).reshape(shape) - 1
    return Population(
        wells=wells, mask=mask, line=line, vendor_gamma=vendor_gamma,
        decline=rng.uniform(0, 15e5, shape) / 365.0,
        wc_rate=rng.uniform(0, 0.0015, shape),
        gor_drift=rng.normal(0, 0.0005, shape),
        erosion=rng.uniform(0, 0.05, shape) / 365.0,
        sal_target=np.where(rng.random(shape) < 0.4, 0.035, fl.salinity),
        sal_rate=rng.uniform(0.0, 0.01, shape),
        meter_sal=fl.salinity.copy(),
        meter_cd=rng.normal(1.0, 0.015, shape),
        meter_d=np.clip(np.sqrt(wells.d_choke * wells.d_tub) * 1.6, 0.05, 0.12),
        test_phase=rng.integers(0, 30, shape),
        well_id=np.where(mask, ids, -1),
    )


def state_on(pop, day):
    """Truth well parameters on a given day."""
    w0 = pop.wells
    f0 = w0.fluid
    fl = pvt.Fluid(api=f0.api, gas_sg=f0.gas_sg, gor=f0.gor * np.exp(pop.gor_drift * day),
                   wc=np.clip(f0.wc + pop.wc_rate * day, 0.0, 0.95),
                   salinity=pop.sal_target + (f0.salinity - pop.sal_target) * (1 - pop.sal_rate) ** day)
    return well.Wells(p_res=np.where(pop.mask, np.maximum(w0.p_res - pop.decline * day, 60e5), w0.p_res),
                      t_res=w0.t_res, pi=w0.pi, depth=w0.depth, d_tub=w0.d_tub, d_choke=w0.d_choke,
                      cd=w0.cd * (1.0 + pop.erosion * day), trim_gamma=w0.trim_gamma, t_sea=w0.t_sea, q_t=w0.q_t,
                      fluid=fl)


def choke_schedule(pop, days, seed=0, p_move=0.1, p_shut=0.005):
    """Operator choke positions per day, shape (days, G, W): piecewise constant moves and short shut-ins."""
    rng = np.random.default_rng(seed + 1)
    g, w = pop.mask.shape
    op = np.empty((days, g, w))
    cur = rng.uniform(0.3, 0.9, (g, w))
    shut = np.zeros((g, w), int)
    for t in range(days):
        move = rng.random((g, w)) < p_move
        cur = np.where(move, np.clip(cur + rng.normal(0, 0.15, (g, w)), 0.15, 1.0), cur)
        start = (shut == 0) & (rng.random((g, w)) < p_shut)
        shut = np.where(start, rng.integers(1, 6, (g, w)), np.maximum(shut - 1, 0))
        op[t] = np.where(shut > 0, 0.0, cur) * pop.mask
    return op


def simulate(pop, days=180, seed=0, leak=None, leak_start=None, log_every=0, n_q=40):
    """Run the field day by day. leak: network.Leak (G,) active from day leak_start (G,) onwards.
    Returns DataFrames: daily (well-days), tests (separator tests), groups (group-days)."""
    rng = np.random.default_rng(seed + 2)
    g, w = pop.mask.shape
    openings = choke_schedule(pop, days, seed)
    rows, test_rows, group_rows = [], [], []
    gi, wi = np.nonzero(pop.mask)
    none = network.Leak.none(g)
    for t in range(days):
        st = state_on(pop, t)
        table = well.vlp_table(st, n_q=n_q)
        lk = none
        if leak is not None:
            active = t >= leak_start
            lk = network.Leak(kind=np.where(active, leak.kind, 0), well=leak.well, pos=leak.pos,
                              d_hole=np.where(active, leak.d_hole, 0.0))
        out = network.solve(st, table, openings[t], pop.line, lk, iters=32)
        info = choke.mass_rate(out['p_wh'], out['p_man'][:, None] * np.ones((1, w)), out['t_wh'], st.fluid,
                               openings[t], st.d_choke, st.cd, st.trim_gamma, info=True)[1]
        cfg = mpfm.Config(salinity=pop.meter_sal, cd_ratio=pop.meter_cd, noise_l=0.02, noise_g=0.04)
        rd = mpfm.read(out['p_wh'], out['t_wh'], st.fluid, out['q_l'], pop.meter_d, cfg, rng=rng)
        flowing = out['q_l'] > 0
        rows.append(pd.DataFrame({
            'day': t, 'group': gi, 'slot': wi, 'well_id': pop.well_id[gi, wi],
            'opening': openings[t][gi, wi], 'd_choke': st.d_choke[gi, wi], 'vendor_gamma': pop.vendor_gamma[gi, wi],
            'p_wh': out['p_wh'][gi, wi], 't_wh': out['t_wh'][gi, wi], 'p_man': out['p_man'][gi],
            'p_sep': pop.line.p_sep[gi],
            'mpfm_q_o': np.where(flowing, rd['q_o'], 0.0)[gi, wi], 'mpfm_q_g': np.where(flowing, rd['q_g'], 0.0)[gi, wi],
            'mpfm_q_w': np.where(flowing, rd['q_w'], 0.0)[gi, wi],
            'true_q_o': out['q_o'][gi, wi], 'true_q_g': out['q_g'][gi, wi], 'true_q_w': out['q_w'][gi, wi],
            'true_m': out['m'][gi, wi], 'true_critical': info['critical'][gi, wi], 'true_y': info['y'][gi, wi],
            'true_gor': st.fluid.gor[gi, wi], 'true_wc': st.fluid.wc[gi, wi], 'true_sal': st.fluid.salinity[gi, wi],
            'hidden_cd': st.cd[gi, wi], 'hidden_trim_gamma': st.trim_gamma[gi, wi],
            'p_res': st.p_res[gi, wi],
        }))
        testing = pop.mask & ((t + pop.test_phase) % 30 == 0) & flowing
        if testing.any():
            tg, tw = np.nonzero(testing)
            noise = lambda: np.exp(rng.normal(0.0, 0.015, len(tg)))
            test_rows.append(pd.DataFrame({'day': t, 'group': tg, 'slot': tw, 'well_id': pop.well_id[tg, tw],
                                           'test_q_o': out['q_o'][tg, tw] * noise(),
                                           'test_q_g': out['q_g'][tg, tw] * noise(),
                                           'test_q_w': out['q_w'][tg, tw] * noise()}))
        group_rows.append(pd.DataFrame({'day': t, 'group': np.arange(g), 'p_man': out['p_man'],
                                        'p_riser_base': out['p_riser_base'], 'p_sep': pop.line.p_sep,
                                        'sep_q_o': out['sep_q_o'], 'sep_q_g': out['sep_q_g'],
                                        'sep_q_w': out['sep_q_w'], 'leak_m': out['leak_m'],
                                        'leak_q_o': out['leak_q_o'], 'leak_q_g': out['leak_q_g'],
                                        'leak_q_w': out['leak_q_w'], 'leak_kind': lk.kind}))
        if log_every and t % log_every == 0:
            print(f'day {t}: flowing {int(flowing[pop.mask].sum())}/{int(pop.mask.sum())}, '
                  f'oil {u.si_to_stb_d(out["sep_q_o"].sum()):.0f} stb/d', flush=True)
    return {'daily': pd.concat(rows, ignore_index=True),
            'tests': pd.concat(test_rows, ignore_index=True) if test_rows else pd.DataFrame(),
            'groups': pd.concat(group_rows, ignore_index=True)}


def wells_frame(pop):
    """Static per-well data: design values a greenfield well has on day 1, plus hidden truth."""
    gi, wi = np.nonzero(pop.mask)
    w, f = pop.wells, pop.wells.fluid
    return pd.DataFrame({
        'well_id': pop.well_id[gi, wi], 'group': gi, 'slot': wi,
        'design_api': f.api[gi, wi], 'design_gas_sg': f.gas_sg[gi, wi], 'design_gor': f.gor[gi, wi],
        'design_wc': f.wc[gi, wi], 'design_sal': f.salinity[gi, wi],
        'd_choke': w.d_choke[gi, wi], 'vendor_gamma': pop.vendor_gamma[gi, wi], 'd_tub': w.d_tub[gi, wi],
        'depth': w.depth[gi, wi], 'meter_d': pop.meter_d[gi, wi],
        'hidden_cd0': w.cd[gi, wi], 'hidden_trim_gamma': w.trim_gamma[gi, wi], 'hidden_pi': w.pi[gi, wi],
        'hidden_p_res0': w.p_res[gi, wi], 'line_length': pop.line.length[gi], 'line_depth': pop.line.depth[gi],
    })


def summarize(res):
    d = res['daily']
    fl = d[d.true_m > 0]
    q_l = u.si_to_stb_d(fl.true_q_o + fl.true_q_w)
    mp_err = (fl.mpfm_q_o + fl.mpfm_q_w) / (fl.true_q_o + fl.true_q_w) - 1
    pct = lambda s: {k: float(np.percentile(s, k)) for k in (1, 10, 50, 90, 99)}
    return {
        'well_days': int(len(d)), 'flowing_fraction': float(len(fl) / max(len(d), 1)),
        'wells': int(d.well_id.nunique()), 'groups': int(d.group.nunique()), 'days': int(d.day.nunique()),
        'liquid_stb_d': pct(q_l), 'p_wh_bar': pct(fl.p_wh / 1e5), 'p_man_bar': pct(fl.p_man / 1e5),
        'pressure_ratio_y': pct(fl.true_y), 'critical_fraction': float(fl.true_critical.mean()),
        'water_cut': pct(fl.true_wc), 'gor_scf_stb': pct(u.si_to_scf_stb(fl.true_gor)),
        'mpfm_liquid_rel_error': pct(mp_err), 'tests': int(len(res['tests'])),
    }


In [ ]:
%%writefile src/vfmchoke/units.py
"""Unit conversions. Everything inside the package is SI (Pa, K, kg, m, s); field units appear only at the edges."""
import numpy as np

G = 9.80665                      # m/s2
PSI = 6894.757293168             # Pa per psi
BAR = 1.0e5
P_SC = 101325.0                  # Pa, standard pressure (14.696 psia)
T_SC = 288.705556                # K, standard temperature (60 F)
BBL = 0.158987294928             # m3 per barrel
SCF = 0.028316846592             # m3 per cubic foot
DAY = 86400.0
INCH = 0.0254
R_GAS = 8314.462618              # J/(kmol K)
M_AIR = 28.9647                  # kg/kmol
RHO_AIR_SC = P_SC * M_AIR / (R_GAS * T_SC)     # ~1.2228 kg/m3
SCF_PER_STB = SCF / BBL          # Sm3/Sm3 in one scf/stb (~0.1781)
RHO_SEAWATER = 1025.0            # kg/m3


def f_to_k(t_f):
    return (np.asarray(t_f, dtype=float) - 32.0) * 5.0 / 9.0 + 273.15


def k_to_f(t_k):
    return (np.asarray(t_k, dtype=float) - 273.15) * 9.0 / 5.0 + 32.0


def c_to_k(t_c):
    return np.asarray(t_c, dtype=float) + 273.15


def k_to_r(t_k):
    return np.asarray(t_k, dtype=float) * 1.8


def pa_to_psia(p):
    return np.asarray(p, dtype=float) / PSI


def psia_to_pa(p):
    return np.asarray(p, dtype=float) * PSI


def pa_to_psig(p):
    return pa_to_psia(p) - P_SC / PSI


def psig_to_pa(p):
    return psia_to_pa(np.asarray(p, dtype=float) + P_SC / PSI)


def api_to_sg(api):
    return 141.5 / (np.asarray(api, dtype=float) + 131.5)


def sg_to_api(sg):
    return 141.5 / np.asarray(sg, dtype=float) - 131.5


def scf_stb_to_si(r):
    """Gas-oil or gas-liquid ratio, scf/stb -> Sm3/Sm3."""
    return np.asarray(r, dtype=float) * SCF_PER_STB


def si_to_scf_stb(r):
    return np.asarray(r, dtype=float) / SCF_PER_STB


def stb_d_to_si(q):
    """Liquid rate, stb/d -> Sm3/s."""
    return np.asarray(q, dtype=float) * BBL / DAY


def si_to_stb_d(q):
    return np.asarray(q, dtype=float) * DAY / BBL


def choke64_to_m(s64):
    """Choke bean size in 64ths of an inch -> diameter in metres."""
    return np.asarray(s64, dtype=float) / 64.0 * INCH


def m_to_choke64(d):
    return np.asarray(d, dtype=float) / INCH * 64.0


In [ ]:
%%writefile src/vfmchoke/well.py
"""Reservoir inflow and vertical tubing lift for a set of wells (all arrays, vectorised over wells).

Inflow: linear productivity above the bubble point, Vogel below (composite IPR), on stock-tank liquid.
Lift: a vertical tubing traverse marched upward with drift-flux gas holdup (C0 = 1.2, Taylor-bubble drift
velocity) and homogeneous friction. Each well gets a lookup table of wellhead pressure and temperature against
liquid rate, and its operating point is where the choke passes exactly what the well delivers.
"""
from dataclasses import dataclass, fields

import numpy as np

from . import choke, pvt, units as u

PI_FIELD = u.BBL / u.DAY / u.PSI        # one stb/d/psi in Sm3/s/Pa


@dataclass
class Wells:
    """Per-well parameters, all arrays of the same shape."""
    p_res: np.ndarray       # reservoir pressure, Pa
    t_res: np.ndarray       # reservoir temperature, K
    pi: np.ndarray          # productivity index on stock-tank liquid, Sm3/s/Pa
    depth: np.ndarray       # true vertical depth of the tubing, m
    d_tub: np.ndarray       # tubing inner diameter, m
    d_choke: np.ndarray     # choke bean diameter at full opening, m
    cd: np.ndarray          # choke discharge coefficient
    trim_gamma: np.ndarray  # choke trim exponent (effective area = opening**gamma)
    t_sea: np.ndarray       # seabed / ambient temperature, K
    q_t: np.ndarray         # rate scale of wellhead heating, Sm3/s
    fluid: pvt.Fluid
    k_mult: np.ndarray = None   # choke-rate multiplier (a calibrated VFM: exp(f + b0)); truth wells use 1
    k_exp: np.ndarray = None    # extra opening exponent (the VFM latent b1); truth wells use 0

    def __post_init__(self):
        if self.k_mult is None:
            self.k_mult = np.ones_like(np.asarray(self.p_res, dtype=float))
        if self.k_exp is None:
            self.k_exp = np.zeros_like(np.asarray(self.p_res, dtype=float))
        for f in fields(self):
            if f.name != 'fluid':
                setattr(self, f.name, np.asarray(getattr(self, f.name), dtype=float))

    def _vfm_terms(self):
        """(k_mult, k_exp), defaulting for objects pickled before these fields existed."""
        km, ke = getattr(self, 'k_mult', None), getattr(self, 'k_exp', None)
        shape = np.shape(self.p_res)
        return (np.ones(shape) if km is None else km), (np.zeros(shape) if ke is None else ke)

    def choke_rate(self, p1, p2, t1, opening, expand=False):
        """Mass rate through each well's choke (with the VFM multiplier and exponent when set)."""
        ex = (lambda a: np.asarray(a)[..., None]) if expand else (lambda a: a)
        fl = self.fluid.expand() if expand else self.fluid
        km, ke = self._vfm_terms()
        m = choke.mass_rate(p1, p2, t1, fl, opening, ex(self.d_choke), ex(self.cd), ex(self.trim_gamma))
        return m * ex(km) * np.maximum(np.asarray(opening, dtype=float), 1e-3) ** ex(ke)

    @property
    def p_b(self):
        return pvt.bubble_point(self.t_res, self.fluid)

    def take(self, idx):
        km, ke = self._vfm_terms()
        kw = {f.name: np.asarray(getattr(self, f.name))[idx] for f in fields(self)
              if f.name not in ('fluid', 'k_mult', 'k_exp')}
        return Wells(fluid=self.fluid.take(idx), k_mult=np.asarray(km)[idx], k_exp=np.asarray(ke)[idx], **kw)


def ipr_rate(p_wf, p_res, p_b, pi):
    """Stock-tank liquid rate (Sm3/s) at bottomhole flowing pressure p_wf."""
    p_wf = np.minimum(p_wf, p_res)
    pb = np.minimum(p_b, p_res)
    q_b = pi * (p_res - pb)
    r = np.clip(p_wf / np.maximum(pb, 1.0), 0.0, 1.0)
    below = q_b + pi * pb / 1.8 * (1.0 - 0.2 * r - 0.8 * r ** 2)
    return np.where(p_wf >= pb, pi * (p_res - p_wf), below)


def aof(p_res, p_b, pi):
    """Absolute open flow: the rate at zero bottomhole pressure."""
    pb = np.minimum(p_b, p_res)
    return pi * (p_res - pb) + pi * pb / 1.8


def ipr_pwf(q, p_res, p_b, pi):
    """Inverse of ipr_rate: bottomhole pressure that delivers liquid rate q."""
    pb = np.minimum(p_b, p_res)
    q_b = pi * (p_res - pb)
    above = p_res - q / pi
    s = np.clip(1.0 - 1.8 * (q - q_b) / np.maximum(pi * pb, 1e-30), 0.0, 1.0)
    r = (-0.2 + np.sqrt(0.04 + 3.2 * s)) / 1.6
    return np.where(q <= q_b, above, np.maximum(r, 0.0) * pb)


def wellhead_temperature(q, t_res, t_sea, q_t):
    """Flowing wellhead temperature: hotter with rate, approaching reservoir temperature at high rates."""
    return t_sea + (t_res - t_sea) * q / (q + q_t)


def friction_factor(re, rel_rough):
    """Darcy friction factor: laminar below Re 2300, Swamee-Jain above."""
    re = np.maximum(re, 1.0)
    turb = 0.25 / np.log10(rel_rough / 3.7 + 5.74 / re ** 0.9) ** 2
    return np.where(re < 2300.0, 64.0 / re, turb)


def gas_holdup(v_sg, v_sl, rho_l, rho_g, d, c0=1.2):
    """Drift-flux gas holdup in vertical upflow (Taylor-bubble drift velocity)."""
    v_m = v_sg + v_sl
    v_d = 0.35 * np.sqrt(u.G * d * np.maximum(rho_l - rho_g, 0.0) / rho_l)
    alpha = np.where(v_sg > 0, v_sg / np.maximum(c0 * v_m + v_d, 1e-12), 0.0)
    return np.clip(alpha, 0.0, 0.999)


def gradient(p, t, q_l, fluid, d, rough, vertical=1.0):
    """Pressure gradient (Pa/m) of upward flow: hydrostatic (times the vertical fraction) plus friction."""
    st = pvt.state(p, t, fluid)
    area = np.pi * d ** 2 / 4.0
    v_sl = q_l * st.v_l / area
    v_sg = q_l * st.v_g / area
    alpha = gas_holdup(v_sg, v_sl, st.rho_l, st.rho_g, d)
    rho_mix = alpha * st.rho_g + (1.0 - alpha) * st.rho_l
    v_m = v_sl + v_sg
    lam = np.where(v_m > 0, v_sl / np.maximum(v_m, 1e-12), 1.0)
    rho_ns = lam * st.rho_l + (1.0 - lam) * st.rho_g
    mu_ns = lam * st.mu_l + (1.0 - lam) * st.mu_g
    f = friction_factor(rho_ns * v_m * d / mu_ns, rough / d)
    return rho_mix * u.G * vertical + f * rho_ns * v_m ** 2 / (2.0 * d)


def traverse_up(p_bottom, q_l, fluid, t_bottom, t_top, depth, d_tub, n_seg=24, rough=4.5e-5):
    """Wellhead pressure for liquid rate q_l flowing up a vertical tubing from bottomhole pressure p_bottom."""
    p = np.array(np.broadcast_to(p_bottom, np.broadcast(p_bottom, q_l).shape), dtype=float)
    dz = depth / n_seg
    for i in range(n_seg):
        t = t_bottom + (t_top - t_bottom) * (i + 0.5) / n_seg
        p = np.maximum(p - gradient(p, t, q_l, fluid, d_tub, rough) * dz, u.P_SC)
    return p


def vlp_table(w, n_q=48):
    """Per-well table on a uniform liquid-rate grid from 0 to 99.9% of the open-flow potential.
    Returns arrays of shape (..., n_q): q (Sm3/s), p_wh (Pa), t_wh (K)."""
    q_max = 0.999 * aof(w.p_res, w.p_b, w.pi)
    q = q_max[..., None] * np.linspace(0.0, 1.0, n_q)
    ex = lambda a: np.asarray(a)[..., None]
    p_wf = ipr_pwf(q, ex(w.p_res), ex(w.p_b), ex(w.pi))
    t_wh = wellhead_temperature(q, ex(w.t_res), ex(w.t_sea), ex(w.q_t))
    p_wh = traverse_up(p_wf, q, w.fluid.expand(), ex(w.t_res), t_wh, ex(w.depth), ex(w.d_tub))
    return {'q': q, 'p_wh': p_wh, 't_wh': t_wh}


def interp(table, q):
    """Wellhead pressure and temperature at liquid rate q, interpolated linearly on each well's uniform grid."""
    grid = table['q']
    n = grid.shape[-1]
    q_max = grid[..., -1]
    s = np.clip(np.asarray(q, dtype=float) / np.maximum(q_max, 1e-30), 0.0, 1.0) * (n - 1)
    i0 = np.clip(np.floor(s).astype(int), 0, n - 2)
    w = (s - i0)[..., None]
    idx = i0[..., None]

    def at(tab):
        a = np.take_along_axis(tab, idx, axis=-1)
        b = np.take_along_axis(tab, idx + 1, axis=-1)
        return (a * (1.0 - w) + b * w)[..., 0]

    return at(table['p_wh']), at(table['t_wh'])


def solve_rate(w, table, p2, opening, iters=30):
    """Operating point of each well for downstream (manifold) pressure p2 and choke opening.
    F(q) = choke rate at the wellhead pressure the well delivers at q, minus q. A gassy well's wellhead pressure can
    rise before it falls, giving an unstable low-rate root as well; the stable operating point is the highest-rate
    sign change of F from + to -, which is the one taken here.
    Returns q_l (Sm3/s stock-tank liquid), p_wh, t_wh, m (kg/s); a well that cannot flow returns zero."""
    q = table['q']
    n = q.shape[-1]
    ex = lambda a: np.asarray(a)[..., None]
    m_grid = w.choke_rate(table['p_wh'], ex(p2), table['t_wh'], ex(opening), expand=True)
    f_grid = m_grid / ex(pvt.mass_per_liquid_sc(w.fluid)) - q
    nonneg = f_grid >= 0
    nonneg[..., 0] = f_grid[..., 0] > 0           # F(0) = 0 means the well cannot open against p2
    any_pos = nonneg.any(axis=-1)
    i_last = n - 1 - np.argmax(nonneg[..., ::-1], axis=-1)
    at_max = any_pos & (i_last == n - 1)
    i0 = np.clip(i_last, 0, n - 2)
    lo = np.take_along_axis(q, i0[..., None], axis=-1)[..., 0]
    hi = np.take_along_axis(q, (i0 + 1)[..., None], axis=-1)[..., 0]
    mls = pvt.mass_per_liquid_sc(w.fluid)
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        p_wh, t_wh = interp(table, mid)
        f_mid = w.choke_rate(p_wh, p2, t_wh, opening) / mls - mid
        lo = np.where(f_mid >= 0, mid, lo)
        hi = np.where(f_mid >= 0, hi, mid)
    q_star = np.where(at_max, q[..., -1], 0.5 * (lo + hi))
    q_star = np.where(any_pos, q_star, 0.0)
    p_wh, t_wh = interp(table, q_star)
    return {'q_l': q_star, 'p_wh': p_wh, 't_wh': t_wh, 'm': q_star * mls}


In [ ]:
%%writefile tests/conftest.py
import sys
from pathlib import Path

import numpy as np
import pytest

sys.path.insert(0, str(Path(__file__).resolve().parents[1] / 'src'))

from vfmchoke import network, pvt, units as u, well  # noqa: E402


@pytest.fixture
def fluid():
    return pvt.Fluid(api=35.0, gas_sg=0.75, gor=150.0, wc=0.2, salinity=0.05)


def make_wells(g=2, w=3, seed=0):
    rng = np.random.default_rng(seed)
    shape = (g, w)
    fl = pvt.Fluid(api=rng.uniform(28, 40, shape), gas_sg=rng.uniform(0.65, 0.85, shape),
                   gor=rng.uniform(60, 250, shape), wc=rng.uniform(0.0, 0.5, shape),
                   salinity=rng.uniform(0.02, 0.08, shape))
    return well.Wells(p_res=rng.uniform(250e5, 320e5, shape), t_res=u.c_to_k(rng.uniform(80, 110, shape)),
                      pi=rng.uniform(5, 15, shape) * well.PI_FIELD, depth=rng.uniform(1800, 2500, shape),
                      d_tub=np.full(shape, 0.1), d_choke=np.full(shape, 0.025), cd=rng.uniform(0.75, 0.85, shape),
                      trim_gamma=rng.uniform(0.9, 1.4, shape), t_sea=np.full(shape, u.c_to_k(4.0)),
                      q_t=np.full(shape, 0.002), fluid=fl)


@pytest.fixture
def wells():
    return make_wells()


@pytest.fixture
def line():
    return network.Line(length=np.array([5000.0, 8000.0]), d=np.array([0.2, 0.2]), depth=np.array([100.0, 120.0]),
                        p_sep=np.array([15e5, 15e5]))


In [ ]:
%%writefile tests/test_models.py
"""Model-layer checks: few-shot filter, array prior, tree export parity, VFM multiplier, setpoint inverse, SimField."""
import numpy as np
import pandas as pd

from vfmchoke import control, edge, fewshot, hybrid, network, pvt, simfield, well


def test_latent_prior_only_without_labels():
    lp = fewshot.LatentPrior(tau0=0.1, tau1=0.2)
    x, p, t_last = fewshot.latent_posterior([], [], [], [], lp)
    assert np.allclose(x, 0) and np.allclose(np.diag(p), [0.01, 0.04]) and t_last is None


def test_latent_recovers_truth_with_many_labels():
    rng = np.random.default_rng(0)
    lp = fewshot.LatentPrior(tau0=0.3, tau1=0.3, q0=0.0)
    b = np.array([0.12, -0.08])
    u = rng.uniform(0.2, 1.0, 400)
    phi = np.column_stack([np.ones_like(u), np.log(u)])
    r = phi @ b + rng.normal(0, 0.02, len(u))
    x, p, _ = fewshot.latent_posterior(np.arange(len(u)), phi, r, np.full(len(u), 0.02), lp)
    assert np.allclose(x, b, atol=0.01)


def test_track_moves_toward_observations():
    m, v = fewshot.track([1, 2, 3], [1.0, 1.0, 1.0], [0.1, 0.1, 0.1], 0.0, 1.0, 0.0, [0])
    assert 0.95 < m[0] <= 1.0 and v < 0.01


def test_prior_arrays_match_prior(fluid):
    df = pd.DataFrame({'p_wh': [50e5, 60e5], 'p_man': [20e5, 25e5], 't_wh': [330.0, 340.0],
                       'opening': [0.5, 0.8], 'd_choke': [0.02, 0.025], 'vendor_gamma': [1.0, 1.6]})
    m1, f1 = hybrid.prior(df, fluid)
    m2, x2 = hybrid.prior_arrays(df.p_wh, df.p_man, df.t_wh, df.opening, df.d_choke, df.vendor_gamma, fluid)
    assert np.allclose(m1, m2) and np.allclose(f1.to_numpy(), x2)
    assert list(f1.columns) == hybrid.FEATURES


def test_tree_export_parity():
    import lightgbm as lgb
    rng = np.random.default_rng(1)
    x = rng.normal(size=(2000, 5))
    y = np.sin(x[:, 0]) + x[:, 1] * x[:, 2] + rng.normal(0, 0.1, 2000)
    m = lgb.LGBMRegressor(n_estimators=80, num_leaves=15, verbose=-1).fit(x, y)
    arr = edge.export_trees(m)
    xt = rng.normal(size=(500, 5))
    xt[::17, 3] = np.nan
    assert np.max(np.abs(edge.predict_trees(arr, xt) - m.predict(xt))) < 1e-9


def test_vfm_multiplier_scales_rate(wells):
    table = well.vlp_table(wells)
    base = wells.choke_rate(table['p_wh'][..., 10], np.full(wells.pi.shape, 20e5), table['t_wh'][..., 10],
                            np.full(wells.pi.shape, 0.5))
    w2 = control.cautious(wells, np.full(wells.pi.shape, np.log(1.2)))
    scaled = w2.choke_rate(table['p_wh'][..., 10], np.full(wells.pi.shape, 20e5), table['t_wh'][..., 10],
                           np.full(wells.pi.shape, 0.5))
    assert np.allclose(scaled, 1.2 * base)


def test_setpoint_hits_target_on_its_own_model(wells):
    w1 = wells.take((np.array([0]), np.array([0])))
    w1 = well.Wells(**{f: np.ravel(getattr(w1, f)) for f in ('p_res', 't_res', 'pi', 'depth', 'd_tub', 'd_choke',
                                                             'cd', 'trim_gamma', 't_sea', 'q_t')},
                    fluid=pvt.Fluid(*(np.ravel(getattr(w1.fluid, n)) for n in ('api', 'gas_sg', 'gor', 'wc',
                                                                                'salinity'))))
    table = well.vlp_table(w1)
    q_half = float(well.solve_rate(w1, table, np.array([20e5]), np.array(0.5))['q_l'][0])
    u_set, ok = control.single_well_setpoint(w1, table, np.array([20e5]), q_half)
    assert ok and abs(u_set - 0.5) < 0.01


def test_simfield_small_run_conserves():
    pop = simfield.sample_population(n_groups=3, max_wells=4, seed=5)
    res = simfield.simulate(pop, days=3, seed=5)
    d = res['daily']
    assert len(d) == 3 * int(pop.mask.sum())
    assert d.true_m.ge(0).all() and (d[d.true_m > 0].mpfm_q_o > 0).all()
    g = res['groups'].sort_values(['group', 'day'])
    m_wells = d.groupby(['group', 'day']).true_m.sum().sort_index().to_numpy()
    m_sep = network.std_mass(g.sep_q_o.to_numpy(), g.sep_q_g.to_numpy(), g.sep_q_w.to_numpy(),
                             pvt.Fluid(35.0, 0.7, 0.0, 0.0, 0.0))
    assert len(m_wells) == len(g) and np.all(m_sep > 0)


def test_edge_step_matches_full_solve(wells, line):
    w = wells.take(np.array([0]))
    w = well.Wells(**{f: np.asarray(getattr(w, f))[0] for f in ('p_res', 't_res', 'pi', 'depth', 'd_tub',
                                                               'd_choke', 'cd', 'trim_gamma', 't_sea', 'q_t')},
                   fluid=pvt.Fluid(*(np.asarray(getattr(w.fluid, n))[0] for n in ('api', 'gas_sg', 'gor', 'wc',
                                                                                 'salinity'))))
    tab = well.vlp_table(w)
    u_grid, p2_grid = np.linspace(0.05, 1.0, 20), np.linspace(10e5, 80e5, 30)
    rt = edge.twin_tables(w, tab, u_grid, p2_grid)
    mls = pvt.mass_per_liquid_sc(w.fluid)
    ln = network.Line(length=line.length[:1], d=line.d[:1], depth=line.depth[:1], p_sep=line.p_sep[:1])
    u_chk = np.full(3, 0.6)
    full = network.solve(control.replicate(w, 1), control.replicate_table(tab, 1), u_chk[None, :], ln, iters=40)
    mix = full['sep_fluid']
    m_grid = np.linspace(0, (rt[:, -1, 0] * mls).sum() * 1.1, 120)
    p_req = edge.line_table(ln, pvt.Fluid(*(np.asarray(getattr(mix, n))[0] for n in ('api', 'gas_sg', 'gor', 'wc',
                                                                                     'salinity'))),
                            float(full['t_mix'][0]), m_grid)
    p_edge, q_edge = edge.edge_step(rt, u_grid, p2_grid, mls, m_grid, p_req, u_chk)
    assert abs(p_edge / full['p_man'][0] - 1) < 0.02
    assert abs(q_edge.sum() / full['q_l'][0].sum() - 1) < 0.03


In [ ]:
%%writefile tests/test_physics.py
"""M1 invariants: PVT bounds and bookkeeping, choke limits and monotonicity, IPR inverse, network conservation,
leak signatures, and exact MPFM inversion."""
import numpy as np

from vfmchoke import choke, mpfm, network, pvt, units as u, well


def envelope(n=4000, seed=1):
    rng = np.random.default_rng(seed)
    f = pvt.Fluid(api=rng.uniform(20, 50, n), gas_sg=rng.uniform(0.6, 1.0, n), gor=rng.uniform(20, 400, n),
                  wc=rng.uniform(0, 0.95, n), salinity=rng.uniform(0, 0.15, n))
    return rng.uniform(5e5, 400e5, n), u.c_to_k(rng.uniform(20, 150, n)), f


# ---------- PVT ----------
def test_pvt_bounds():
    p, t, f = envelope()
    st = pvt.state(p, t, f)
    assert np.all(st.rs >= 0) and np.all(st.rs <= f.gor + 1e-9)
    assert np.all(st.bo >= 1.0) and np.all(st.bo < 3.0)
    assert np.all((st.x >= 0) & (st.x <= 1))
    assert np.all(st.mu_l > 0) and np.all(st.mu_g > 0)
    assert np.all((st.rho_l > 500) & (st.rho_l < 1300))
    z = pvt.z_factor(p, t, f.gas_sg)
    assert np.all((z >= 0.3) & (z <= 1.2))


def test_mass_bookkeeping_exact():
    """Liquid plus free gas at any (p, t) carries exactly the standard mass per Sm3 of liquid."""
    p, t, f = envelope()
    st = pvt.state(p, t, f)
    np.testing.assert_allclose(st.m_l + st.m_g, pvt.mass_per_liquid_sc(f), rtol=1e-12)


def test_gas_ideal_limit():
    rho = pvt.gas_density(u.P_SC, u.T_SC, 0.7)
    assert abs(rho / pvt.rho_gas_sc(0.7) - 1) < 0.01


def test_mix_is_mass_exact():
    rng = np.random.default_rng(3)
    f = pvt.Fluid(api=rng.uniform(25, 45, 5), gas_sg=rng.uniform(0.6, 0.9, 5), gor=rng.uniform(50, 300, 5),
                  wc=rng.uniform(0, 0.7, 5), salinity=rng.uniform(0, 0.1, 5))
    q_o, q_g, q_w = choke.std_rates(rng.uniform(5, 20, 5), f)
    mf = pvt.mix(q_o, q_g, q_w, f)
    total = network.std_mass(q_o.sum(), q_g.sum(), q_w.sum(), mf)
    assert abs(total / network.std_mass(q_o, q_g, q_w, f).sum() - 1) < 1e-12


# ---------- choke ----------
def test_liquid_only_is_bernoulli():
    f = pvt.Fluid(api=35.0, gas_sg=0.75, gor=0.0, wc=0.3, salinity=0.04)
    p1, p2, t1, d = 50e5, 20e5, 330.0, 0.02
    m = choke.mass_rate(p1, p2, t1, f, 1.0, d, cd=0.8)
    rho_l = pvt.state(p1, t1, f).rho_l
    expected = 0.8 * np.pi / 4 * d ** 2 * np.sqrt(2 * rho_l * (p1 - p2))
    assert abs(m / expected - 1) < 1e-9


def test_gas_only_critical_ratio():
    k = 1.28
    yc = choke.critical_ratio(1.0, 1e-3, 0.02, k)
    assert abs(yc - (2 / (k + 1)) ** (k / (k - 1))) < 1e-6


def test_critical_ratio_matches_flux_maximiser():
    """The derivative-root bisection finds the same ratio as a direct search for the flux maximum."""
    rng = np.random.default_rng(4)
    x = np.concatenate([rng.uniform(1e-4, 1.0, 3000), [0.0, 1e-6, 1.0]])
    v_l = rng.uniform(1 / 1100, 1 / 600, x.size)
    v_g1 = rng.uniform(1 / 200, 1 / 5, x.size)
    fast = choke.critical_ratio(x, v_l, v_g1)
    ref = choke.critical_ratio_golden(x, v_l, v_g1, iters=80)
    assert np.max(np.abs(fast - ref)) < 1e-6
    f_fast = choke.flux_shape(fast, x, v_l, v_g1)
    f_ref = choke.flux_shape(ref, x, v_l, v_g1)
    assert np.all(f_fast >= f_ref * (1 - 1e-10))


def test_continuity_at_critical(fluid):
    p1, t1 = 60e5, 340.0
    _, info = choke.mass_rate(p1, 0.1 * p1, t1, fluid, 0.7, 0.025, info=True)
    yc = float(info['yc'])
    below = choke.mass_rate(p1, yc * (1 - 1e-7) * p1, t1, fluid, 0.7, 0.025)
    above = choke.mass_rate(p1, yc * (1 + 1e-7) * p1, t1, fluid, 0.7, 0.025)
    assert abs(below / above - 1) < 1e-3


def test_choke_monotone(fluid):
    t1 = 340.0
    opening = np.linspace(0, 1, 101)
    m = choke.mass_rate(60e5, 20e5, t1, fluid, opening, 0.025, trim_gamma=1.3)
    assert np.all(np.diff(m) >= -1e-12 * m.max())
    p2 = np.linspace(1e5, 59.9e5, 200)
    m = choke.mass_rate(60e5, p2, t1, fluid, 0.6, 0.025)
    assert np.all(np.diff(m) <= 1e-12 * m.max())
    p1 = np.linspace(21e5, 200e5, 200)
    m = choke.mass_rate(p1, 20e5, t1, fluid, 0.6, 0.025)
    assert np.all(np.diff(m) >= -1e-12 * m.max())


def test_gilbert_unit_conversion():
    """A = 10.00 with GLR in scf/stb must match Gilbert's original 435 with GLR in Mscf/bbl (within rounding)."""
    p, s, glr = 800.0, 32.0, 900.0
    original = p * s ** 1.89 / (435.0 * (glr / 1000.0) ** 0.546)
    assert abs(choke.gilbert_rate(p, s, glr, 'gilbert') / original - 1) < 0.005


# ---------- well ----------
def test_ipr_inverse(wells):
    w = wells.take((0, 0))
    pb = min(float(w.p_b), float(w.p_res))
    p = np.linspace(0.05 * pb, float(w.p_res), 200)
    q = well.ipr_rate(p, w.p_res, w.p_b, w.pi)
    assert abs(float(well.ipr_rate(w.p_res, w.p_res, w.p_b, w.pi))) < 1e-12
    np.testing.assert_allclose(well.ipr_pwf(q, w.p_res, w.p_b, w.pi), p, rtol=1e-6, atol=1.0)


def test_operating_point_consistent(wells):
    table = well.vlp_table(wells)
    p2 = np.full(wells.pi.shape, 20e5)
    opening = np.full(wells.pi.shape, 0.6)
    op = well.solve_rate(wells, table, p2, opening)
    assert np.all(op['q_l'] > 0)
    m_choke = choke.mass_rate(op['p_wh'], p2, op['t_wh'], wells.fluid, opening, wells.d_choke, wells.cd,
                              wells.trim_gamma)
    q_choke = m_choke / pvt.mass_per_liquid_sc(wells.fluid)
    np.testing.assert_allclose(q_choke, op['q_l'], rtol=1e-3)


def test_rate_rises_with_opening(wells):
    table = well.vlp_table(wells)
    p2 = np.full(wells.pi.shape, 20e5)
    q = [well.solve_rate(wells, table, p2, np.full(wells.pi.shape, o))['q_l'] for o in (0.2, 0.4, 0.6, 0.8, 1.0)]
    assert np.all(np.diff(np.stack(q), axis=0) >= -1e-9)


# ---------- network ----------
def _conservation(out):
    lhs = out['m'].sum(1)
    rhs = out['leak_m'] + out['sep_m']
    return np.abs(lhs - rhs) / lhs


def test_network_no_leak(wells, line):
    table = well.vlp_table(wells)
    out = network.solve(wells, table, np.full(wells.pi.shape, 0.6), line)
    assert np.all(_conservation(out) < 1e-9)
    assert np.all(np.abs(out['p_man'] - out['p_man_implied']) < 5e3)
    assert np.all(out['leak_m'] == 0)
    assert np.all(out['p_man'] > line.p_sep)


def test_jumper_leak_signature(wells, line):
    table = well.vlp_table(wells)
    leak = network.Leak(kind=np.array([network.JUMPER, network.NONE]), well=np.array([1, 0]),
                        pos=np.zeros(2), d_hole=np.array([0.008, 0.0]))
    out = network.solve(wells, table, np.full(wells.pi.shape, 0.6), line, leak)
    assert np.all(_conservation(out) < 1e-9)
    assert out['leak_m'][0] > 0 and out['leak_m'][1] == 0
    f = wells.fluid.take((0, 1))
    assert abs(out['leak_q_g'][0] / out['leak_q_o'][0] / f.gor - 1) < 1e-9
    wc_leak = out['leak_q_w'][0] / (out['leak_q_o'][0] + out['leak_q_w'][0])
    assert abs(wc_leak - f.wc) < 1e-9


def test_flowline_leak_signature(wells, line):
    table = well.vlp_table(wells)
    opening = np.full(wells.pi.shape, 0.6)
    base = network.solve(wells, table, opening, line)
    leak = network.Leak(kind=np.array([network.NONE, network.FLOWLINE]), well=np.zeros(2, int),
                        pos=np.array([0.0, 0.5]), d_hole=np.array([0.0, 0.01]))
    out = network.solve(wells, table, opening, line, leak)
    assert np.all(_conservation(out) < 1e-9)
    assert out['leak_m'][1] > 0
    gor_leak = out['leak_q_g'][1] / out['leak_q_o'][1]
    assert abs(gor_leak / out['sep_fluid'].gor[1] - 1) < 1e-9
    assert out['sep_q_o'][1] < base['sep_q_o'][1]
    assert out['p_man'][1] < base['p_man'][1]        # less flow downstream of the hole: lower manifold pressure


def test_riser_leak_conserves(wells, line):
    table = well.vlp_table(wells)
    leak = network.Leak(kind=np.array([network.RISER, network.MANIFOLD]), well=np.zeros(2, int),
                        pos=np.array([0.3, 0.0]), d_hole=np.array([0.01, 0.006]))
    out = network.solve(wells, table, np.full(wells.pi.shape, 0.6), line, leak)
    assert np.all(_conservation(out) < 1e-9)
    assert np.all(out['leak_m'] > 0)


# ---------- MPFM ----------
def test_mpfm_exact_inversion():
    rng = np.random.default_rng(7)
    n = 500
    alpha = rng.dirichlet([2, 2, 2], n)
    mu = mpfm.mu_linear(rng.uniform(700, 900, n), rng.uniform(1000, 1100, n), rng.uniform(30, 200, n),
                        rng.uniform(0, 0.1, n))
    d = np.full(n, 0.08)
    np.testing.assert_allclose(mpfm.invert(mpfm.attenuation(alpha, mu, d), mu, d), alpha, atol=1e-9)


def test_mpfm_read_noise_free_is_exact(fluid):
    q_l = 0.01
    cfg = mpfm.Config(salinity=fluid.salinity)
    r = mpfm.read(40e5, 330.0, fluid, q_l, 0.08, cfg)
    q_o, q_g, q_w = choke.std_rates(q_l * pvt.mass_per_liquid_sc(fluid), fluid)
    for got, want in ((r['q_o'], q_o), (r['q_g'], q_g), (r['q_w'], q_w)):
        assert abs(got / want - 1) < 1e-9


def test_mpfm_stale_salinity_overreads_water(fluid):
    q_l = 0.01
    fresh_cfg = mpfm.Config(salinity=fluid.salinity - 0.03)
    r = mpfm.read(40e5, 330.0, fluid, q_l, 0.08, fresh_cfg)
    wc_meter = r['q_w'] / (r['q_o'] + r['q_w'])
    assert wc_meter > fluid.wc


In [ ]:
%%writefile benchmarks/__init__.py
# package marker


In [ ]:
%%writefile benchmarks/dabiri.py
"""B2: Dabiri et al. 2024 (Sci Rep 14:6945), 565 wellhead-choke tests from eight literature sources.

Inputs Pwh (psi), choke size Dc (64ths of an inch), GLR (scf/stb); target liquid rate (stb/d). The data ships as a
DOCX supplementary file (CC BY 4.0); this module downloads and parses it without third-party DOCX libraries.
"""
import io
import zipfile
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd

URLS = [
    'https://pmc.ncbi.nlm.nih.gov/articles/instance/10960849/bin/41598_2024_54010_MOESM1_ESM.docx',
    'https://static-content.springer.com/esm/art%3A10.1038%2Fs41598-024-54010-2/MediaObjects/'
    '41598_2024_54010_MOESM1_ESM.docx',
]
W = '{http://schemas.openxmlformats.org/wordprocessingml/2006/main}'


def download():
    import urllib.request
    last = None
    for url in URLS:
        try:
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (research benchmark)'})
            with urllib.request.urlopen(req, timeout=60) as r:
                data = r.read()
            if data[:2] == b'PK':
                return data, url
            last = f'{url}: not a DOCX ({data[:40]!r})'
        except Exception as e:  # try the next mirror
            last = f'{url}: {e}'
    raise RuntimeError(f'could not download the Dabiri supplementary file; last error {last}')


def tables(docx_bytes):
    """All tables in the document as lists of rows of cell strings."""
    root = ET.fromstring(zipfile.ZipFile(io.BytesIO(docx_bytes)).read('word/document.xml'))
    out = []
    for tbl in root.iter(W + 'tbl'):
        rows = []
        for tr in tbl.iter(W + 'tr'):
            rows.append([''.join(t.text or '' for t in tc.iter(W + 't')).strip() for tc in tr.iter(W + 'tc')])
        out.append(rows)
    return out


def _num(s):
    s = s.replace(',', '').replace('−', '-').strip()
    try:
        return float(s)
    except ValueError:
        return np.nan


def parse(docx_bytes):
    """Rows with four numeric values in the order the paper uses (Pwh, Dc, GLR, Qliq), plus any text label that
    looks like a source reference. Column order is checked against the paper's ranges."""
    recs, headers = [], []
    for ti, rows in enumerate(tables(docx_bytes)):
        for row in rows:
            nums = [_num(c) for c in row]
            vals = [v for v in nums if np.isfinite(v)]
            text = [c for c, v in zip(row, nums) if not np.isfinite(v) and c]
            if len(vals) >= 4:
                recs.append({'table': ti, 'vals': vals, 'text': ' '.join(text)})
            elif text:
                headers.append((ti, row))
    if not recs:
        raise RuntimeError('no numeric rows found in the DOCX tables')
    n_cols = pd.Series([len(r['vals']) for r in recs]).mode().iloc[0]
    arr = np.array([r['vals'][-n_cols:] for r in recs if len(r['vals']) >= n_cols])
    return arr, recs, headers


def assign_columns(arr):
    """Pick the columns that match the paper's ranges: Pwh 50-4308 psi, Dc 14-80 /64 in, GLR 36-13095 scf/stb,
    Qliq 198-25878 stb/d. Returns a DataFrame and the mapping used."""
    ranges = {'pwh': (40, 5000), 'd64': (10, 90), 'glr': (30, 14000), 'qliq': (150, 30000)}
    cols = list(range(arr.shape[1]))
    best, best_score = None, -1
    from itertools import permutations
    for perm in permutations(cols, 4):
        score = 0
        for name, c in zip(ranges, perm):
            lo, hi = ranges[name]
            score += np.mean((arr[:, c] >= lo) & (arr[:, c] <= hi))
        # Dc is the only small-integer-like column
        score += np.mean(np.abs(arr[:, perm[1]] - np.round(arr[:, perm[1]])) < 1e-6) * 0.5
        if score > best_score:
            best, best_score = perm, score
    df = pd.DataFrame({name: arr[:, c] for name, c in zip(ranges, best)})
    return df, dict(zip(ranges, best)), best_score


In [ ]:
%%writefile benchmarks/volve.py
"""B1: Equinor Volve daily production data (Equinor Open Data Licence), loaded from the Kaggle copy.

Per well and day: on-stream hours, wellhead pressure and temperature, choke opening (%), pressure drop over the
choke, and allocated oil, gas and water volumes. Rates are volumes scaled to a full day.
"""
from pathlib import Path

import numpy as np
import pandas as pd

PRODUCERS = ['NO 15/9-F-1 C', 'NO 15/9-F-11 H', 'NO 15/9-F-12 H', 'NO 15/9-F-14 H', 'NO 15/9-F-15 D']
FOUR = ['NO 15/9-F-11 H', 'NO 15/9-F-12 H', 'NO 15/9-F-14 H', 'NO 15/9-F-15 D']


def find_xlsx(root='/kaggle/input'):
    hits = sorted(Path(root).rglob('*olve*roduction*.xlsx'))
    if not hits:
        raise FileNotFoundError('Volve production workbook not found under ' + root)
    return hits[0]


def load(path=None, min_hours=20.0):
    path = path or find_xlsx()
    raw = pd.read_excel(path, sheet_name='Daily Production Data')
    raw = raw[raw.WELL_BORE_CODE.isin(PRODUCERS) & (raw.FLOW_KIND == 'production')].copy()
    raw['date'] = pd.to_datetime(raw.DATEPRD)
    hrs = raw.ON_STREAM_HRS.astype(float)
    full = 24.0 / hrs.where(hrs > 0)
    df = pd.DataFrame({
        'well': raw.WELL_BORE_CODE, 'date': raw.date, 'hours': hrs,
        'opening': raw.AVG_CHOKE_SIZE_P.astype(float) / 100.0,
        'p_wh_barg': raw.AVG_WHP_P.astype(float), 't_wh_c': raw.AVG_WHT_P.astype(float),
        'dp_choke_bar': raw.DP_CHOKE_SIZE.astype(float),
        'q_o': raw.BORE_OIL_VOL.astype(float) * full, 'q_g': raw.BORE_GAS_VOL.astype(float) * full,
        'q_w': raw.BORE_WAT_VOL.astype(float) * full,
    })
    ok = ((df.hours >= min_hours) & (df.q_o > 0) & (df.opening > 0.01) & (df.p_wh_barg > 1)
          & (df.dp_choke_bar > 0) & (df.dp_choke_bar < df.p_wh_barg) & df.t_wh_c.between(1, 150))
    df = df[ok].sort_values(['well', 'date']).reset_index(drop=True)
    df['day'] = (df.date - df.groupby('well').date.transform('min')).dt.days
    return df


In [ ]:
# M1 gate: run the physics invariants and keep the report as a kernel output.
import json
import os
import subprocess
import sys
import time

t0 = time.time()
env = dict(os.environ, PYTHONPATH='src')
r = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'tests', '-rA',
                    '--junitxml=test_report.xml'], capture_output=True, text=True, env=env)
print(r.stdout[-30000:])
print(r.stderr[-5000:])
lines = r.stdout.strip().splitlines()
summary = {'returncode': r.returncode, 'seconds': round(time.time() - t0, 1), 'summary': lines[-1] if lines else ''}
with open('test_report.json', 'w') as fh:
    json.dump(summary, fh, indent=1)
print(summary)
